# MNIST Full SpikeEngine Scratchpad

This notebook is the MNIST SpikeEngine experiment rebuilt on the spikecorec engine. The reservoir is a sparse small-world torus on GPU with low-rank heterogeneous recurrent weights, and a feature sampler.

The probe sees only engine state: spike-recency trace and membrane voltage. Pixels are injected only as input current into configured input neurons.


In [ ]:
# import gzip
# import importlib
# import math
# import struct
# import time
from dataclasses import dataclass, replace

import mnist

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm



import spikecorec as spc
from spikecorec import small_world_torus, square_torus, random_fixed_outdegree
from spikecorec import SpikeEngine

print(f"numpy {np.__version__}; spc {spc.__version__}")

## MNIST Loader

The loader is dependency-light: it downloads the raw IDX files once into `.spikecore.cache/mnist` and parses them directly.

In [ ]:
# MNIST_CACHE = Path(".spikecore.cache/mnist")
# MNIST_BASE_URL = "https://storage.googleapis.com/cvdf-datasets/mnist/"
MNIST_FILES = {
    "train_images": "train-images-idx3-ubyte.gz",
    "train_labels": "train-labels-idx1-ubyte.gz",
    "test_images": "t10k-images-idx3-ubyte.gz",
    "test_labels": "t10k-labels-idx1-ubyte.gz",
}


def load_mnist():
    return (
        mnist.train_images(),
        mnist.train_labels(),
        mnist.test_images(),
        mnist.test_labels()
    )

def get_random_balanced_subset(images, labels, per_digit: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    indices = []
    for digit in range(10):
        digit_index = np.flatnonzero(labels == digit)
        indices.extend(rng.choice(digit_index, size=per_digit, replace=False).tolist())
    rng.shuffle(indices)
    return images[indices], labels[indices]

def get_random_data_sample(images, labels, count: int, rng):
    index = rng.integers(0, len(labels), size=count)
    return images[index], labels[index]

def input_from_images(images, input_shape=None):
    images = np.asarray(images)
    if images.ndim != 3:
        raise ValueError(f"images must have shape (n, height, width), got {images.shape}")

    # FLAG: why is config not being passed directly instead of querying globals? 
    # FLAG: the experiment config should have a method that gets the input shape and if no shape is configured just by default return (14, 14) and then pass that method call into this 
    #       function or call that method in this function from passed in config
    if input_shape is None:
        config = globals().get("ENGINE_CONFIG", None)
        input_shape = getattr(config, "input_shape", (14, 14))

    rows, cols = (int(input_shape[0]), int(input_shape[1]))
    if rows <= 0 or cols <= 0:
        raise ValueError(f"input_shape must be positive, got {input_shape}")

    height, width = images.shape[1], images.shape[2]
    if rows == height and cols == width:
        x = images.astype(np.float32) / 255.0

    elif rows == 14 and cols == 14 and height == 28 and width == 28:
        # Preserve the original default mapping: every other MNIST pixel.
        x = images[:, ::2, ::2].astype(np.float32) / 255.0

    elif height % rows == 0 and width % cols == 0:
        row_factor = height // rows
        col_factor = width // cols
        x = images.reshape(len(images), rows, row_factor, cols, col_factor).mean(axis=(2, 4)).astype(np.float32) / 255.0

    else:
        row_idx = np.linspace(0, height - 1, rows).round().astype(np.int64)
        col_idx = np.linspace(0, width - 1, cols).round().astype(np.int64)
        x = images[:, row_idx][:, :, col_idx].astype(np.float32) / 255.0

    return np.ascontiguousarray(x.reshape(len(images), rows * cols), dtype=np.float32)


def one_hot_encoding(labels, classes: int = 10):
    y = np.zeros((len(labels), classes), dtype=np.float32)
    y[np.arange(len(labels)), np.asarray(labels, dtype=np.int64)] = 1.0
    return y

train_images_all, train_labels_all, test_images_all, test_labels_all = load_mnist()
print(train_images_all.shape, test_images_all.shape)


## Engine Configuration

The 14x14 MNIST input is mapped into a 28x28 square-torus spike engine. Each low-resolution pixel drives the center neuron of its corresponding 2x2 reservoir block. Recurrent weights are held near the single-step bifurcation estimate by using the engine's constant-weight mode; set `freeze_learning=False` to allow the engine STDP path to update weights.

In [ ]:
@dataclass
class EngineMNISTConfig:
    input_side: int = 14
    reservoir_scale: int = 2
    input_shape: tuple[int, int] | None = None
    reservoir_side: int | None = None
    input_mapping: str = "auto"
    rank: int = 64
    resting_mp: float = 0.1
    spike_threshold: float = 1.0
    decay_rate: float = 0.22
    recurrent_scale: float = 1.03
    freeze_learning: bool = True
    input_gain: float = 1.15
    pre_steps: int = 3
    on_steps: int = 32
    off_steps: int = 8
    readout_start: int = 12
    readout_stride: int = 4
    spike_tau: float = 10.0
    feature_voltage_scale: float = 1.0
    weight_init_scale: float = 0.01
    topology: str = "random_fixed_outdegree"
    random_fanout: int = 8
    seed: int = 7
    use_batched_feature_extraction: bool = True
    feature_batch_size: int = 128

    def __post_init__(self):
        if self.input_shape is None:
            self.input_shape = (self.input_side, self.input_side)
        else:
            if len(self.input_shape) != 2:
                raise ValueError(f"input_shape must be a 2D tuple, got {self.input_shape}")
            self.input_side = int(self.input_shape[0])

        if self.input_shape[0] < 1 or self.input_shape[1] < 1:
            raise ValueError(f"input_shape entries must be >= 1, got {self.input_shape}")

        self.reservoir_scale = max(1, int(self.reservoir_scale))
        if self.reservoir_side is None:
            self.reservoir_side = max(self.input_shape) * self.reservoir_scale

        if self.reservoir_side < 1:
            raise ValueError(f"reservoir_side must be >= 1, got {self.reservoir_side}")

        self.input_mapping = str(self.input_mapping).lower()
        if self.input_mapping not in {"auto", "grid", "random"}:
            raise ValueError("input_mapping must be one of: auto, grid, random")

        self.rank = max(1, int(self.rank))
        self.random_fanout = max(0, int(self.random_fanout))
        self.pre_steps = max(0, int(self.pre_steps))
        self.on_steps = max(1, int(self.on_steps))
        self.off_steps = max(0, int(self.off_steps))
        self.readout_start = max(1, int(self.readout_start))
        self.readout_stride = max(1, int(self.readout_stride))
        self.feature_batch_size = max(1, int(self.feature_batch_size))


@dataclass
class ProbeConfig:
    train_per_digit: int = 500
    test_per_digit: int = 100
    batch_size: int = 128
    ridge: float = 1e-2
    rls_delta: float = 1e-2
    rls_forgetting: float = 1.0
    rls_epochs: int = 1
    rls_batch_size: int = 64
    online_lr: float = 5e-2
    online_l2: float = 1e-4
    online_epochs: int = 3
    online_batch_size: int = 64


@dataclass
class RLMNISTConfig:
    train_per_digit: int = 50
    test_per_digit: int = 20
    epochs: int = 5
    batch_size: int = 64
    eval_every: int = 1
    lr: float = 5e-2
    l2: float = 1e-5
    evidence_gain: float = 0.35
    decision_threshold: float = 1.0
    elimination_threshold: float = -1.0
    temperature: float = 0.20
    exploration_noise: float = 0.03
    correct_reward: float = 1.0
    wrong_penalty: float = -1.0
    timeout_penalty: float = -0.35
    time_penalty: float = 0.01
    dense_reward_scale: float = 0.02
    channel_shaping_scale: float = 0.5
    baseline_decay: float = 0.95
    readout_exclude_input: bool = True
    readout_neighborhood_hops: int = 0
    seed: int = 23


ENGINE_CONFIG = EngineMNISTConfig(input_side=28)
PROBE_CONFIG = ProbeConfig()
RL_MNIST_CONFIG = RLMNISTConfig()
print(ENGINE_CONFIG)


## SpikeEngine MNIST Reservoir

This wrapper owns no dynamics of its own. It only resets the engine, injects encoded input current into configured input neurons, and reads membrane/spike-recency features from the engine state.

In [ ]:
class SpikeEngineMNISTReservoir:
    frame_title = "SpikeEngine membrane potential"

    def __init__(self, config: EngineMNISTConfig):
        self.config = config
        self.rng = np.random.default_rng(config.seed)
        self.n_input = int(np.prod(config.input_shape))
        self.n_reservoir = config.reservoir_side * config.reservoir_side
        layout = self._make_input_layout()
        self.input_neurons = np.asarray(layout["input_neurons"], dtype=np.int32)
        self.input_mapping_mode = layout["mode"]
        self.raw_input_to_engine_slot = layout["raw_input_to_engine_slot"]
        self.input_projection_counts = layout["input_projection_counts"]
        self.n_engine_input = int(len(self.input_projection_counts))
        np.random.seed(config.seed)
        network = self._make_network()
        self.engine = SpikeEngine(
            network,
            (config.reservoir_side, config.reservoir_side),
            rank=config.rank,
            resting_mp=config.resting_mp,
            decay_rate=config.decay_rate,
            # weight_initializer=lambda size: cp.random.normal(0.0, config.weight_init_scale, size=size),
        )
        self.engine.spike_threshold = np.float32(config.spike_threshold)
        self.engine.set_input_neurons(self.input_neurons)
        self.weight_summary = self.engine.scale_randomized_weights_near_bifurcation(
            input_period=1,
            scale=config.recurrent_scale,
            freeze_learning=config.freeze_learning,
        )
        self._feature_buffer = np.empty((2 * self.n_reservoir + 1,), dtype=np.float32)
        # self._batched_step_kernel = None
        # self._batched_feature_kernel = None
        # self._input_slot_by_neuron = None
        self.reset()

    @property
    def n_features(self):
        return 2 * self.n_reservoir

    def _make_network(self):
        side = int(self.config.reservoir_side)
        n = side * side
        fanout = max(0, int(self.config.random_fanout))
        if self.config.topology == "square_torus" or fanout == 0 or n <= 1:
            return square_torus(side)

        if self.config.topology == "small_world_torus":
            base_children = square_torus(side)[0]
            used = set(int(x) for x in base_children)
            used.add(0)
            max_extra = max(0, n - len(used))
            return small_world_torus(side, random_fanout=min(fanout, max_extra), seed=self.config.seed)

        if self.config.topology in {"random_fixed_outdegree", "random"}:
            return random_fixed_outdegree(side, fanout=min(fanout, max(1, n - 1)), seed=self.config.seed)

        raise ValueError(f"Unsupported topology: {self.config.topology}")

    def _make_input_layout(self):
        side = int(self.config.reservoir_side)
        rows, columns = (int(self.config.input_shape[0]), int(self.config.input_shape[1]))
        mapping = str(getattr(self.config, "input_mapping", "auto")).lower()
        grid_ok = rows <= side and columns <= side and side % rows == 0 and side % columns == 0
        if mapping != "random" and grid_ok:
            row_scale = side // rows
            col_scale = side // columns
            row_offset = row_scale // 2
            col_offset = col_scale // 2
            ids = [
                (row_scale * r + row_offset) * side + (col_scale * c + col_offset)
                for r in range(rows)
                for c in range(columns)
            ]
            return {
                "input_neurons": np.asarray(ids, dtype=np.int32),
                "raw_input_to_engine_slot": np.arange(self.n_input, dtype=np.int32),
                "input_projection_counts": np.ones(self.n_input, dtype=np.float32),
                "mode": "grid",
            }

        rng = np.random.default_rng(int(self.config.seed) + 1009)
        engine_inputs = min(self.n_input, self.n_reservoir)
        input_neurons = rng.choice(self.n_reservoir, size=engine_inputs, replace=False).astype(np.int32)
        if self.n_input <= engine_inputs:
            raw_to_slot = np.arange(self.n_input, dtype=np.int32)
        else:
            raw_to_slot = np.empty(self.n_input, dtype=np.int32)
            raw_to_slot[:engine_inputs] = rng.permutation(engine_inputs).astype(np.int32)
            raw_to_slot[engine_inputs:] = rng.integers(0, engine_inputs, size=self.n_input - engine_inputs, dtype=np.int32)
            rng.shuffle(raw_to_slot)
        counts = np.bincount(raw_to_slot, minlength=engine_inputs).astype(np.float32)
        counts = np.maximum(counts, 1.0).astype(np.float32)
        return {
            "input_neurons": input_neurons,
            "raw_input_to_engine_slot": raw_to_slot,
            "input_projection_counts": counts,
            "mode": "random" if mapping == "random" else "random_fallback",
        }

    def _prepare_input_vectors(self, input_vectors):
        X = np.asarray(input_vectors, dtype=np.float32)
        if X.ndim == 1:
            X = X[None, :]
        if X.ndim != 2:
            raise ValueError(f"input_vectors must have shape (n, {self.n_input}), got {X.shape}")
        if X.shape[1] != self.n_input:
            raise ValueError(
                f"input vector width {X.shape[1]} does not match config.input_shape={self.config.input_shape} "
                f"({self.n_input} values). Use input_from_images(images, {self.config.input_shape}) or rebuild the reservoir."
            )
        if not np.isfinite(X).all():
            raise ValueError("input_vectors contain NaN or Inf values")
        return np.ascontiguousarray(X, dtype=np.float32)

    def _project_prepared_input_vectors(self, input_vectors):
        X = np.asarray(input_vectors, dtype=np.float32)
        if self.n_engine_input == self.n_input and np.array_equal(self.raw_input_to_engine_slot, np.arange(self.n_input, dtype=np.int32)):
            return np.ascontiguousarray(X, dtype=np.float32)
        projected = np.zeros((len(X), self.n_engine_input), dtype=np.float32)
        row_idx = np.repeat(np.arange(len(X), dtype=np.int64), self.n_input)
        slot_idx = np.tile(self.raw_input_to_engine_slot.astype(np.int64), len(X))
        np.add.at(projected, (row_idx, slot_idx), X.reshape(-1))
        projected /= self.input_projection_counts[None, :]
        return np.ascontiguousarray(projected, dtype=np.float32)

    def _project_input_vectors(self, input_vectors):
        return self._project_prepared_input_vectors(self._prepare_input_vectors(input_vectors))

    def _prepare_input_vector(self, input_vector):
        return self._prepare_input_vectors(input_vector)[0]

    def _project_input_vector(self, input_vector):
        return self._project_input_vectors(input_vector)[0]

    def input_layout_summary(self):
        collisions = int(self.n_input - self.n_engine_input)
        return {
            "mode": self.input_mapping_mode,
            "raw_inputs": int(self.n_input),
            "engine_inputs": int(self.n_engine_input),
            "reservoir_neurons": int(self.n_reservoir),
            "collisions": max(0, collisions),
        }

    def input_neuron_indices(self):
        return np.asnumpy(self.input_neurons).astype(np.int64)

    def readout_feature_mask(self, exclude_input: bool = True, neighborhood_hops: int = 0):
        excluded = set()
        if exclude_input:
            excluded.update(int(i) for i in self.input_neuron_indices())
        # hops = int(neighborhood_hops)
        # if hops > 0 and excluded:
        #     graph = getattr(self.engine.weights, "network", None)
        #     if graph is None:
        #         raise RuntimeError("Neighborhood masking requires the reservoir graph to be stored.")
        #     frontier = set(excluded)
        #     for _ in range(hops):
        #         next_frontier = set()
        #         for src, children in graph.items():
        #             src = int(src)
        #             child_set = {int(c) for c in children}
        #             if src in frontier:
        #                 next_frontier.update(child_set)
        #             if frontier.intersection(child_set):
        #                 next_frontier.add(src)
        #         next_frontier.difference_update(excluded)
        #         excluded.update(next_frontier)
        #         frontier = next_frontier
        #         if not frontier:
        #             break

        neuron_mask = np.ones(self.n_reservoir, dtype=bool)
        if excluded:
            valid = np.asarray([i for i in excluded if 0 <= i < self.n_reservoir], dtype=np.int64)
            neuron_mask[valid] = False
        feature_mask = np.concatenate([neuron_mask, neuron_mask]).astype(bool)
        if not feature_mask.any():
            return np.ones(2 * self.n_reservoir, dtype=bool)
        return feature_mask

    def reset(self):
        self.engine.reset_state()
        self.tick = 2
        self.sequence_step = 0

    def apply_runtime_controls(
        self,
        *,
        input_gain: float | None = None,
        recurrent_scale: float | None = None,
        decay_rate: float | None = None,
        spike_threshold: float | None = None,
        resting_mp: float | None = None,
        spike_tau: float | None = None,
        feature_voltage_scale: float | None = None,
        reset_state: bool = False,
    ):
        if input_gain is not None:
            self.config.input_gain = float(input_gain)
        if decay_rate is not None:
            self.config.decay_rate = float(decay_rate)
            self.engine.decay_rate = np.float32(self.config.decay_rate)
        if spike_threshold is not None:
            self.config.spike_threshold = float(spike_threshold)
            self.engine.spike_threshold = np.float32(self.config.spike_threshold)
        if resting_mp is not None:
            old_rest = float(self.engine.resting_mp)
            self.config.resting_mp = float(resting_mp)
            self.engine.RESTING_MP = np.float32(self.config.resting_mp)
            self.engine.membrane_potentials += np.float32(self.config.resting_mp - old_rest)
        if spike_tau is not None:
            self.config.spike_tau = float(spike_tau)
        if feature_voltage_scale is not None:
            self.config.feature_voltage_scale = float(feature_voltage_scale)
        if recurrent_scale is not None:
            self.config.recurrent_scale = float(recurrent_scale)
            self.weight_summary = self.engine.scale_random_weights_near_bifurcation(
                input_period=1,
                scale=self.config.recurrent_scale,
                freeze_learning=self.config.freeze_learning,
            )
        if reset_state:
            self.reset()
        return self.weight_summary

    def envelope(self, step: int):
        if step < self.config.pre_steps:
            return 0.0
        if step < self.config.pre_steps + self.config.on_steps:
            return 1.0
        return 0.0

    def step(self, input_vector, sequence_step: int | None = None):
        step_index = self.sequence_step if sequence_step is None else int(sequence_step)
        values = self._project_input_vector(input_vector) * (self.config.input_gain * self.envelope(step_index))
        self.engine.advance_static_input(values, self.tick, self.input_neurons, full_decay=False)
        self.tick += 1
        self.sequence_step = step_index + 1

    def _snapshot_feature_device(self):
        features = self.engine.reservoir_features_device(
            self.tick,
            spike_tau=max(1e-6, self.config.spike_tau),
            voltage_scale=max(1e-6, self.config.feature_voltage_scale),
            out=self._feature_buffer,
        )
        return features[:self.n_features]

    def _snapshot_feature(self):
        return np.asnumpy(self._snapshot_feature_device()).astype(np.float32)

    def _compile_batched_kernels(self):
        if self._batched_step_kernel is not None and self._batched_feature_kernel is not None:
            return
        src = r"""
        __device__ __forceinline__ float batched_apply_decay(float mp, const float rest, const float decay) {
            return rest + (mp - rest) * (1.0f - decay);
        }

        extern "C" __global__
        void mnist_batched_step_kernel(
            int batch_size,
            int neuron_count,
            int tick,
            int spike_period,
            float spike_threshold,
            float decay_rate,
            float resting_mp,
            const float* __restrict__ U,
            const float* __restrict__ V,
            int use_constant_weight,
            float constant_weight,
            const int* __restrict__ neighbors,
            const float* __restrict__ input_vectors,
            const int* __restrict__ input_slot_by_neuron,
            int input_dim,
            float input_scale,
            const float* __restrict__ current_inputs,
            float* __restrict__ next_inputs,
            float* __restrict__ membrane,
            int* __restrict__ last_spiked
        ) {
            int idx = blockDim.x * blockIdx.x + threadIdx.x;
            int total = batch_size * neuron_count;
            if (idx >= total) return;
            int b = idx / neuron_count;
            int neuron = idx - b * neuron_count;

            float mp = batched_apply_decay(membrane[idx], resting_mp, decay_rate);
            float total_input = current_inputs[idx];
            int slot = input_slot_by_neuron[neuron];
            if (slot >= 0 && slot < input_dim) {
                total_input += input_vectors[b * input_dim + slot] * input_scale;
            }
            mp += total_input;

            int previous_spike = last_spiked[idx];
            if ((tick - previous_spike) == spike_period) {
                membrane[idx] = resting_mp;
                return;
            }

            if (mp > spike_threshold) {
                if ((tick - previous_spike) > spike_period) {
                    last_spiked[idx] = tick;
                }
                int base = neuron * <<NEIGHB_COUNT_SUB>>;
                for (int c = 0; c < <<NEIGHB_COUNT_SUB>>; ++c) {
                    int child = neighbors[base + c];
                    float weight = constant_weight;
                    if (!use_constant_weight) {
                        const float* u = U + (size_t)neuron * <<K_SUB>>;
                        const float* v = V + (size_t)child * <<K_SUB>>;
                        float dot = 0.0f;
                        for (int d = 0; d < <<K_SUB>>; ++d) {
                            dot += u[d] * v[d];
                        }
                        weight = dot;
                    }
                    atomicAdd(&next_inputs[b * neuron_count + child], weight);
                }
            }
            membrane[idx] = mp;
        }

        extern "C" __global__
        void mnist_batched_feature_kernel(
            int batch_size,
            int neuron_count,
            int feature_count,
            int readout_index,
            int tick,
            float spike_tau,
            float voltage_scale,
            const float* __restrict__ membrane,
            const int* __restrict__ last_spiked,
            float resting_mp,
            float* __restrict__ features
        ) {
            int idx = blockDim.x * blockIdx.x + threadIdx.x;
            int total = batch_size * neuron_count;
            if (idx >= total) return;
            int b = idx / neuron_count;
            int neuron = idx - b * neuron_count;
            int base = (b * feature_count + readout_index) * (2 * neuron_count);

            int last = last_spiked[idx];
            float trace = 0.0f;
            if (last > 0) {
                int age = tick - last;
                if (age < 0) age = 0;
                trace = expf(-(float)age / spike_tau);
            }
            features[base + neuron] = trace;
            features[base + neuron_count + neuron] = (membrane[idx] - resting_mp) / voltage_scale;
        }
        """
        src = src.replace("<<NEIGHB_COUNT_SUB>>", str(int(self.engine.weights.neighb_count)))
        src = src.replace("<<K_SUB>>", str(int(self.engine.weights.U.shape[1])))
        self._batched_step_kernel = cp.RawKernel(src, "mnist_batched_step_kernel")
        self._batched_feature_kernel = cp.RawKernel(src, "mnist_batched_feature_kernel")

    def _input_slot_by_neuron_device(self):
        if self._input_slot_by_neuron is None:
            slots = cp.full((self.n_reservoir,), -1, dtype=cp.int32)
            slots[self.input_neurons] = cp.arange(self.n_engine_input, dtype=cp.int32)
            self._input_slot_by_neuron = slots
        return self._input_slot_by_neuron

    def _readout_after_steps(self):
        cfg = self.config
        total_steps = int(cfg.pre_steps + cfg.on_steps + cfg.off_steps)
        steps = [
            step + 1
            for step in range(total_steps)
            if (step + 1) >= int(cfg.readout_start)
            and ((step + 1) - int(cfg.readout_start)) % int(cfg.readout_stride) == 0
        ]
        return steps if steps else [total_steps]

    def _collect_feature_sequences_batched(self, input_vectors, batch_size: int = 128, progress_desc: str = "feature sequences"):
        cfg = self.config
        self._compile_batched_kernels()
        input_vectors = self._prepare_input_vectors(input_vectors)
        readout_after_steps = self._readout_after_steps()
        readout_lookup = {int(step): idx for idx, step in enumerate(readout_after_steps)}
        readout_count = len(readout_after_steps)
        out = np.empty((len(input_vectors), readout_count, self.n_features), dtype=np.float32)
        mask = np.ones((len(input_vectors), readout_count), dtype=bool)
        if len(input_vectors) == 0:
            return out, mask

        configured_batch = int(getattr(cfg, "feature_batch_size", batch_size))
        batch_size = max(1, int(batch_size if batch_size is not None else configured_batch))
        batch_size = min(batch_size, max(1, configured_batch))
        threads = 256
        input_slots = self._input_slot_by_neuron_device()
        neighbors = self.engine.weights.neighbors.reshape(-1)
        use_constant_weight = int(bool(self.engine.use_constant_weight or self.engine.weights.use_constant_weight))
        constant_weight = self.engine.weights.constant_weight
        if constant_weight is None:
            constant_weight = cp.float32(0.0)
        total_steps = int(cfg.pre_steps + cfg.on_steps + cfg.off_steps)

        starts = range(0, len(input_vectors), batch_size)
        if progress_desc:
            starts = tqdm(starts, desc=progress_desc, leave=False)
        for start in starts:
            stop = min(start + batch_size, len(input_vectors))
            chunk = cp.asarray(self._project_prepared_input_vectors(input_vectors[start:stop]), dtype=cp.float32)
            current_batch = int(stop - start)
            state_shape = (current_batch, self.n_reservoir)
            membrane = cp.empty(state_shape, dtype=cp.float32)
            membrane.fill(cp.float32(cfg.resting_mp))
            last_spiked = cp.zeros(state_shape, dtype=cp.int32)
            current_inputs = cp.zeros(state_shape, dtype=cp.float32)
            next_inputs = cp.zeros(state_shape, dtype=cp.float32)
            features = cp.empty((current_batch, readout_count, self.n_features), dtype=cp.float32)
            blocks = (current_batch * self.n_reservoir + threads - 1) // threads

            for step in range(total_steps):
                next_inputs.fill(cp.float32(0.0))
                tick = 2 + step
                input_scale = float(cfg.input_gain * self.envelope(step))
                self._batched_step_kernel(
                    (blocks,), (threads,),
                    (
                        cp.int32(current_batch),
                        cp.int32(self.n_reservoir),
                        cp.int32(tick),
                        cp.int32(1),
                        cp.float32(cfg.spike_threshold),
                        cp.float32(cfg.decay_rate),
                        cp.float32(cfg.resting_mp),
                        self.engine.weights.U,
                        self.engine.weights.V,
                        cp.int32(use_constant_weight),
                        constant_weight,
                        neighbors,
                        chunk,
                        input_slots,
                        cp.int32(self.n_engine_input),
                        cp.float32(input_scale),
                        current_inputs,
                        next_inputs,
                        membrane,
                        last_spiked,
                    ),
                )
                current_inputs, next_inputs = next_inputs, current_inputs
                after_step = step + 1
                if after_step in readout_lookup:
                    readout_idx = readout_lookup[after_step]
                    self._batched_feature_kernel(
                        (blocks,), (threads,),
                        (
                            cp.int32(current_batch),
                            cp.int32(self.n_reservoir),
                            cp.int32(readout_count),
                            cp.int32(readout_idx),
                            cp.int32(tick + 1),
                            cp.float32(max(1e-6, cfg.spike_tau)),
                            cp.float32(max(1e-6, cfg.feature_voltage_scale)),
                            membrane,
                            last_spiked,
                            cp.float32(cfg.resting_mp),
                            features,
                        ),
                    )
            out[start:stop] = cp.asnumpy(features)
        return out.astype(np.float32), mask

    def collect_features(self, input_vectors, batch_size: int = 128, progress_desc: str = "features"):
        cfg = self.config
        if bool(getattr(cfg, "use_batched_feature_extraction", True)):
            sequences, mask = self._collect_feature_sequences_batched(input_vectors, batch_size=batch_size, progress_desc=progress_desc)
            denom = np.maximum(1, mask.sum(axis=1)).astype(np.float32)[:, None]
            return (sequences.sum(axis=1) / denom).astype(np.float32)
        input_vectors = self._prepare_input_vectors(input_vectors)
        total_steps = cfg.pre_steps + cfg.on_steps + cfg.off_steps
        out = np.empty((len(input_vectors), self.n_features), dtype=np.float32)
        starts = range(0, len(input_vectors), batch_size)
        if progress_desc:
            starts = tqdm(starts, desc=progress_desc, leave=False)
        for start in starts:
            stop = min(start + batch_size, len(input_vectors))
            for row, x in enumerate(input_vectors[start:stop], start=start):
                self.reset()
                accum = cp.zeros(self.n_features, dtype=cp.float32)
                count = 0
                for step in range(total_steps):
                    self.step(x, sequence_step=step)
                    if self.sequence_step >= cfg.readout_start and (self.sequence_step - cfg.readout_start) % cfg.readout_stride == 0:
                        accum += self._snapshot_feature_device()
                        count += 1
                out[row] = cp.asnumpy(accum / max(1, count)).astype(np.float32)
        return out

    def collect_feature_sequence(self, input_vector):
        input_vector = self._prepare_input_vector(input_vector)
        cfg = self.config
        total_steps = cfg.pre_steps + cfg.on_steps + cfg.off_steps
        features = []
        self.reset()
        for step in range(total_steps):
            self.step(input_vector, sequence_step=step)
            if self.sequence_step >= cfg.readout_start and (self.sequence_step - cfg.readout_start) % cfg.readout_stride == 0:
                features.append(self._snapshot_feature())
        if not features:
            features.append(self._snapshot_feature())
        return np.stack(features, axis=0).astype(np.float32)

    def collect_feature_sequences(self, input_vectors, batch_size: int = 64, progress_desc: str = "feature sequences"):
        cfg = self.config
        if bool(getattr(cfg, "use_batched_feature_extraction", True)):
            return self._collect_feature_sequences_batched(input_vectors, batch_size=batch_size, progress_desc=progress_desc)
        input_vectors = self._prepare_input_vectors(input_vectors)
        sequences = []
        starts = range(0, len(input_vectors), batch_size)
        if progress_desc:
            starts = tqdm(starts, desc=progress_desc, leave=False)
        for start in starts:
            stop = min(start + batch_size, len(input_vectors))
            for x in input_vectors[start:stop]:
                sequences.append(self.collect_feature_sequence(x))
        max_steps = max(seq.shape[0] for seq in sequences)
        out = np.zeros((len(sequences), max_steps, self.n_features), dtype=np.float32)
        mask = np.zeros((len(sequences), max_steps), dtype=bool)
        for idx, seq in enumerate(sequences):
            out[idx, : seq.shape[0]] = seq
            mask[idx, : seq.shape[0]] = True
        return out, mask

    def current_frame(self):
        self.engine._decay_all(self.tick)
        return cp.asnumpy(self.engine.membrane_potentials).reshape(self.config.reservoir_side, self.config.reservoir_side)

    def stats(self):
        frame = self.current_frame()
        last = cp.asnumpy(self.engine.last_spiked).astype(np.int32)
        recent = np.mean((self.tick - last) <= 1)
        ever = np.mean(last > 0)
        return {
            "mp_mean": float(frame.mean()),
            "mp_std": float(frame.std()),
            "mp_max": float(frame.max()),
            "recent_spike_fraction": float(recent),
            "ever_spiked_fraction": float(ever),
        }


reservoir = SpikeEngineMNISTReservoir(ENGINE_CONFIG)
assert reservoir.n_features == 2 * reservoir.n_reservoir
assert not reservoir.engine.use_constant_weight
print("input neurons:", reservoir.n_input, "reservoir neurons:", reservoir.n_reservoir, "features:", reservoir.n_features)
print("input layout:", reservoir.input_layout_summary())
print("readout features: spike trace + membrane voltage")
print("weight summary:", reservoir.weight_summary)
print("k2tree constructed:", reservoir.engine.weights.k2tree is not None)


## Linear Readouts

The ridge probe is the closed-form baseline. The RLS probe is the online readout used for incremental training.

In [ ]:
def _finite_feature_matrix(features, *, name="features"):
    X = np.asarray(features, dtype=np.float32)
    if X.ndim == 1:
        X = X[None, :]
    if X.ndim != 2:
        raise ValueError(f"{name} must be a 2D feature matrix, got {X.shape}")
    if not np.isfinite(X).all():
        raise ValueError(f"{name} contain NaN or Inf values; check reservoir/input_shape alignment and dynamics controls.")
    return X


def _feature_mean_std(features):
    X = _finite_feature_matrix(features)
    mean = X.mean(axis=0).astype(np.float32)
    std = X.std(axis=0).astype(np.float32)
    std = np.where(np.isfinite(std) & (std > 1e-6), std, 1.0).astype(np.float32)
    return X, mean, std


class RidgeProbe:
    def __init__(self, weights, mean, std):
        self.weights = weights.astype(np.float32)
        self.mean = mean.astype(np.float32)
        self.std = std.astype(np.float32)
        self.readout_name = "batch ridge probe"

    @classmethod
    def fit(cls, features, labels, ridge: float = 1e-2):
        X, mean, std = _feature_mean_std(features)
        Xn = (X - mean) / std
        X_aug = np.column_stack([Xn, np.ones(len(Xn), dtype=np.float32)])
        Y = one_hot(labels)
        lhs = X_aug.T @ X_aug
        lhs.flat[:: lhs.shape[0] + 1] += float(ridge)
        lhs[-1, -1] -= float(ridge)
        rhs = X_aug.T @ Y
        try:
            weights = np.linalg.solve(lhs, rhs)
        except np.linalg.LinAlgError:
            jitter = max(float(ridge), 1e-6) * 10.0
            lhs.flat[:: lhs.shape[0] + 1] += jitter
            weights = np.linalg.solve(lhs, rhs)
        return cls(weights, mean, std)

    def logits(self, features):
        X = _finite_feature_matrix(features)
        Xn = (X - self.mean) / self.std
        X_aug = np.column_stack([Xn, np.ones(len(Xn), dtype=np.float32)])
        return X_aug @ self.weights

    def probabilities(self, features):
        z = self.logits(features)
        z = z - z.max(axis=1, keepdims=True)
        e = np.exp(z)
        return e / np.maximum(1e-8, e.sum(axis=1, keepdims=True))

    def predict(self, features):
        return self.logits(features).argmax(axis=1)


class OnlineRLSProbe:
    """Mini-batch recursive least-squares readout for online reservoir training."""

    def __init__(self, n_features: int, delta: float = 1e-2, forgetting: float = 1.0):
        self.n_features = int(n_features)
        self.delta = float(delta)
        self.forgetting = float(forgetting)
        self.mean = np.zeros(self.n_features, dtype=np.float32)
        self.std = np.ones(self.n_features, dtype=np.float32)
        self.readout_name = "online RLS probe"
        self.reset_weights()

    def set_normalizer(self, features):
        X, self.mean, self.std = _feature_mean_std(features)

    def reset_weights(self, delta: float | None = None):
        if delta is not None:
            self.delta = float(delta)
        prior = max(float(self.delta), 1e-8)
        self.weights = np.zeros((self.n_features + 1, 10), dtype=np.float32)
        self.P = np.eye(self.n_features + 1, dtype=np.float32) / prior
        self.updates = 0
        self.examples_seen = 0
        self.last_loss = None
        self.last_batch_acc = None

    def _standardize(self, features):
        X = _finite_feature_matrix(features)
        return (X - self.mean) / self.std

    def _augment(self, features):
        X = self._standardize(features)
        return np.column_stack([X, np.ones(len(X), dtype=np.float32)]).astype(np.float32)

    def logits(self, features):
        return self._augment(features) @ self.weights

    def probabilities(self, features):
        z = self.logits(features)
        z = z - z.max(axis=1, keepdims=True)
        e = np.exp(z)
        return e / np.maximum(1e-8, e.sum(axis=1, keepdims=True))

    def predict(self, features):
        return self.logits(features).argmax(axis=1)

    def partial_fit(self, features, labels, forgetting: float | None = None):
        labels = np.asarray(labels, dtype=np.int64).reshape(-1)
        X = self._augment(features)
        Y = one_hot(labels).astype(np.float32)
        lam = self.forgetting if forgetting is None else float(forgetting)
        lam = float(np.clip(lam, 0.90, 1.0))
        self.forgetting = lam

        predictions_before = X @ self.weights
        error = Y - predictions_before
        XP = X @ self.P
        S = XP @ X.T
        S.flat[:: S.shape[0] + 1] += lam
        try:
            K_T = np.linalg.solve(S, XP)
        except np.linalg.LinAlgError:
            jitter = 1e-5 * max(1.0, float(np.trace(S)) / max(1, S.shape[0]))
            S.flat[:: S.shape[0] + 1] += jitter
            K_T = np.linalg.solve(S, XP)

        self.weights += (K_T.T @ error).astype(np.float32)
        self.P = ((self.P - K_T.T @ XP) / lam).astype(np.float32)
        self.P = (0.5 * (self.P + self.P.T)).astype(np.float32)

        predictions_after = X @ self.weights
        self.last_loss = float(np.mean((Y - predictions_after) ** 2))
        self.last_batch_acc = float(np.mean(predictions_after.argmax(axis=1) == labels))
        self.updates += 1
        self.examples_seen += len(labels)
        return self.last_loss, self.last_batch_acc

    def train_epochs(self, features, labels, epochs: int = 1, batch_size: int = 128, seed: int = 0, progress_desc: str | None = None):
        rng = np.random.default_rng(seed)
        labels = np.asarray(labels, dtype=np.int64)
        history = []
        epoch_iter = range(int(epochs))
        if progress_desc is not None:
            epoch_iter = tqdm(epoch_iter, total=int(epochs), desc=progress_desc, leave=False)
        for epoch in epoch_iter:
            order = rng.permutation(len(labels))
            batch_iter = range(0, len(order), int(batch_size))
            if progress_desc is not None:
                batch_iter = tqdm(batch_iter, total=(len(order) + int(batch_size) - 1) // int(batch_size), desc=f"{progress_desc} epoch {epoch + 1}", leave=False)
            for start in batch_iter:
                batch_idx = order[start:start + int(batch_size)]
                self.partial_fit(features[batch_idx], labels[batch_idx])
            history.append((epoch + 1, self.last_loss, self.last_batch_acc))
        return history


class OnlineSoftmaxProbe:
    def __init__(self, n_features: int, lr: float = 5e-2, l2: float = 1e-4):
        self.n_features = int(n_features)
        self.lr = float(lr)
        self.l2 = float(l2)
        self.mean = np.zeros(self.n_features, dtype=np.float32)
        self.std = np.ones(self.n_features, dtype=np.float32)
        self.weights = np.zeros((self.n_features + 1, 10), dtype=np.float32)
        self.updates = 0
        self.examples_seen = 0
        self.last_loss = None
        self.last_batch_acc = None
        self.readout_name = "online softmax probe"

    def set_normalizer(self, features):
        X, self.mean, self.std = _feature_mean_std(features)

    def reset_weights(self):
        self.weights.fill(0.0)
        self.updates = 0
        self.examples_seen = 0
        self.last_loss = None
        self.last_batch_acc = None

    def _augment(self, features):
        X = _finite_feature_matrix(features)
        X = (X - self.mean) / self.std
        return np.column_stack([X, np.ones(len(X), dtype=np.float32)]).astype(np.float32)

    def logits(self, features):
        return self._augment(features) @ self.weights

    def probabilities(self, features):
        z = self.logits(features)
        z = z - z.max(axis=1, keepdims=True)
        e = np.exp(z)
        return e / np.maximum(1e-8, e.sum(axis=1, keepdims=True))

    def predict(self, features):
        return self.logits(features).argmax(axis=1)

    def partial_fit(self, features, labels, lr: float | None = None, l2: float | None = None):
        labels = np.asarray(labels, dtype=np.int64).reshape(-1)
        X = self._augment(features)
        step_lr = self.lr if lr is None else float(lr)
        step_l2 = self.l2 if l2 is None else float(l2)
        logits = X @ self.weights
        logits = logits - logits.max(axis=1, keepdims=True)
        probs = np.exp(logits)
        probs /= np.maximum(1e-8, probs.sum(axis=1, keepdims=True))
        Y = one_hot(labels)
        grad = X.T @ (probs - Y) / len(labels)
        grad[:-1] += step_l2 * self.weights[:-1]
        self.weights -= step_lr * grad.astype(np.float32)
        self.last_loss = float(-(Y * np.log(probs + 1e-8)).sum(axis=1).mean())
        self.last_batch_acc = float(np.mean(probs.argmax(axis=1) == labels))
        self.updates += 1
        self.examples_seen += len(labels)
        return self.last_loss, self.last_batch_acc

    def train_epochs(self, features, labels, epochs: int = 1, batch_size: int = 128, seed: int = 0, progress_desc: str | None = None):
        rng = np.random.default_rng(seed)
        labels = np.asarray(labels, dtype=np.int64)
        history = []
        epoch_iter = range(int(epochs))
        if progress_desc is not None:
            epoch_iter = tqdm(epoch_iter, total=int(epochs), desc=progress_desc, leave=False)
        for epoch in epoch_iter:
            order = rng.permutation(len(labels))
            batch_iter = range(0, len(order), int(batch_size))
            if progress_desc is not None:
                batch_iter = tqdm(batch_iter, total=(len(order) + int(batch_size) - 1) // int(batch_size), desc=f"{progress_desc} epoch {epoch + 1}", leave=False)
            for start in batch_iter:
                batch_idx = order[start:start + int(batch_size)]
                self.partial_fit(features[batch_idx], labels[batch_idx])
            history.append((epoch + 1, self.last_loss, self.last_batch_acc))
        return history


def softmax_accuracy(probe, features, labels):
    return float(np.mean(probe.predict(features) == np.asarray(labels)))


accuracy = softmax_accuracy


## Feature Extraction And Baselines

The defaults match the earlier MNIST scratchpad scale. Lower `train_per_digit` for quick iteration; raise it when you want a stronger readout estimate.

In [ ]:
train_images, train_labels = select_balanced(train_images_all, train_labels_all, PROBE_CONFIG.train_per_digit, seed=1)
test_images, test_labels = select_balanced(test_images_all, test_labels_all, PROBE_CONFIG.test_per_digit, seed=2)

X_train_input = input_from_images(train_images)
X_test_input = input_from_images(test_images)

start = time.perf_counter()
train_features = reservoir.collect_features(X_train_input, batch_size=PROBE_CONFIG.batch_size, progress_desc="train features")
test_features = reservoir.collect_features(X_test_input, batch_size=PROBE_CONFIG.batch_size, progress_desc="test features")
feature_seconds = time.perf_counter() - start
print(f"feature extraction seconds: {feature_seconds:.1f}")
print("feature stats:", train_features.mean(), train_features.std(), train_features.min(), train_features.max())

ridge_probe = RidgeProbe.fit(train_features, train_labels, ridge=PROBE_CONFIG.ridge)
ridge_train_acc = accuracy(ridge_probe, train_features, train_labels)
ridge_test_acc = accuracy(ridge_probe, test_features, test_labels)
print(f"ridge train/test: {ridge_train_acc:.3f} / {ridge_test_acc:.3f}")

rls_probe = OnlineRLSProbe(reservoir.n_features, delta=PROBE_CONFIG.rls_delta, forgetting=PROBE_CONFIG.rls_forgetting)
rls_probe.set_normalizer(train_features)
rls_history = rls_probe.train_epochs(
    train_features,
    train_labels,
    epochs=PROBE_CONFIG.rls_epochs,
    batch_size=PROBE_CONFIG.rls_batch_size,
    seed=3,
    progress_desc="online RLS",
)
rls_train_acc = accuracy(rls_probe, train_features, train_labels)
rls_test_acc = accuracy(rls_probe, test_features, test_labels)
print(f"RLS train/test: {rls_train_acc:.3f} / {rls_test_acc:.3f}")
print(f"RLS updates: {rls_probe.updates}; examples seen: {rls_probe.examples_seen}; last loss: {rls_probe.last_loss:.6f}")

online_probe = OnlineSoftmaxProbe(reservoir.n_features, lr=PROBE_CONFIG.online_lr, l2=PROBE_CONFIG.online_l2)
online_probe.set_normalizer(train_features)
online_history = online_probe.train_epochs(
    train_features,
    train_labels,
    epochs=PROBE_CONFIG.online_epochs,
    batch_size=PROBE_CONFIG.online_batch_size,
    seed=4,
    progress_desc="online softmax",
)
online_train_acc = accuracy(online_probe, train_features, train_labels)
online_test_acc = accuracy(online_probe, test_features, test_labels)
print(f"online softmax train/test: {online_train_acc:.3f} / {online_test_acc:.3f}")
print(f"softmax updates: {online_probe.updates}; examples seen: {online_probe.examples_seen}; last loss: {online_probe.last_loss:.6f}")

mnist_training_history = {
    "updates": [int(rls_probe.updates)],
    "batch_acc": [float(rls_probe.last_batch_acc) if rls_probe.last_batch_acc is not None else np.nan],
    "eval_acc": [float(rls_test_acc)],
    "loss": [float(rls_probe.last_loss) if rls_probe.last_loss is not None else np.nan],
}

## Confusion Matrices

In [ ]:
def confusion_matrix_counts(labels, predictions, classes: int = 10):
    matrix = np.zeros((classes, classes), dtype=np.int64)
    for y, yhat in zip(labels, predictions):
        matrix[int(y), int(yhat)] += 1
    return matrix


def plot_confusion_matrices(probe=rls_probe):
    train_cm = confusion_matrix_counts(train_labels, probe.predict(train_features))
    test_cm = confusion_matrix_counts(test_labels, probe.predict(test_features))
    fig = make_subplots(rows=1, cols=2, subplot_titles=("train", "test"))
    fig.add_trace(go.Heatmap(z=train_cm, colorscale="Blues", showscale=False), row=1, col=1)
    fig.add_trace(go.Heatmap(z=test_cm, colorscale="Blues", showscale=True), row=1, col=2)
    fig.update_layout(width=850, height=380, title=f"Confusion matrices: {probe.readout_name}")
    fig.update_xaxes(title="predicted")
    fig.update_yaxes(title="label", autorange="reversed")
    fig.show()
    return train_cm, test_cm

train_confusion, test_confusion = plot_confusion_matrices(rls_probe)

## Single Digit Probe

Use this cell to inspect the engine response and readout probabilities for individual images.

In [ ]:
def run_digit_example(index: int = 0, source: str = "test", probe=rls_probe):
    images, labels = (test_images_all, test_labels_all) if source == "test" else (train_images_all, train_labels_all)
    image = images[index]
    label = int(labels[index])
    x = input_from_images(image[None, ...])[0]
    feature = reservoir.collect_features(x[None, ...], batch_size=1, progress_desc="single digit")
    probs = probe.probabilities(feature)[0]
    frame = reservoir.current_frame()

    fig = make_subplots(rows=1, cols=3, subplot_titles=(f"image label={label}", "engine membrane", "readout"), specs=[[{}, {}, {"type": "bar"}]])
    fig.add_trace(go.Heatmap(z=image, colorscale="Gray", showscale=False), row=1, col=1)
    fig.add_trace(go.Heatmap(z=frame, colorscale="Viridis", showscale=False), row=1, col=2)
    fig.add_trace(go.Bar(x=list(range(10)), y=probs), row=1, col=3)
    fig.update_layout(width=1050, height=360, title=f"prediction={int(np.argmax(probs))} confidence={float(np.max(probs)):.3f}")
    fig.show()
    return probs
probs = run_digit_example(0, "test")

## Supervised Baseline Live Controls

This UI is kept as a supervised ridge/RLS baseline and dynamics inspector. The preferred RL-style MNIST workflow is the evidence-accumulator workbench below.


## RL-Style Evidence Readout

This readout treats MNIST as a sequential decision problem. The SpikeEngine reservoir is stepped through the image presentation, a zero-centered linear readout produces ten signed drift channels in `[-1, 1]`, and evidence starts at zero. Evidence is accumulated with hard `[-1, 1]` bounds: `+1` selects a digit, while `-1` eliminates that digit from later choices. The reward combines correct/incorrect terminal reward, a timeout penalty, a per-step time cost, and dense shaping from positive correct drift minus positive wrong-digit drift.


In [ ]:

def _softmax_np(logits, temperature: float = 1.0):
    z = np.asarray(logits, dtype=np.float32) / max(1e-6, float(temperature))
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / np.maximum(1e-8, e.sum(axis=-1, keepdims=True))


def _masked_softmax_np(logits, active, temperature: float = 1.0):
    active = np.asarray(active, dtype=bool)
    if not active.any():
        return np.full_like(np.asarray(logits, dtype=np.float32), 0.1, dtype=np.float32)
    masked = np.where(active, np.asarray(logits, dtype=np.float32), -1e9)
    return _softmax_np(masked, temperature=temperature).astype(np.float32)


class EvidenceAccumulatorPolicy:
    def __init__(self, n_features: int, config: RLMNISTConfig):
        self.n_features = int(n_features)
        self.config = replace(config)
        self.rng = np.random.default_rng(config.seed)
        self.mean = np.zeros(self.n_features, dtype=np.float32)
        self.std = np.ones(self.n_features, dtype=np.float32)
        self.W = self.rng.normal(0.0, 1e-3, size=(self.n_features, 10)).astype(np.float32)
        self.b = np.zeros(10, dtype=np.float32)
        self.baseline = 0.0
        self.updates = 0
        self.examples_seen = 0
        self.last_metrics = {}

    def set_normalizer(self, sequences, mask=None):
        X = np.asarray(sequences, dtype=np.float32)
        if X.ndim == 3:
            if mask is not None:
                X = X[np.asarray(mask, dtype=bool)]
            else:
                X = X.reshape(-1, X.shape[-1])
        if X.ndim != 2 or len(X) == 0:
            raise ValueError(f"normalizer needs a non-empty 2D feature matrix, got {X.shape}")
        if not np.isfinite(X).all():
            raise ValueError("evidence normalizer received NaN or Inf features; check input_shape/reservoir size and dynamics controls.")
        self.mean = X.mean(axis=0).astype(np.float32)
        std = X.std(axis=0).astype(np.float32)
        self.std = np.where(np.isfinite(std) & (std > 1e-6), std, 1.0).astype(np.float32)

    def _standardize_sequence(self, sequence, mask=None):
        X = np.asarray(sequence, dtype=np.float32)
        if mask is not None:
            X = X[np.asarray(mask, dtype=bool)]
        if X.ndim != 2:
            raise ValueError("sequence must have shape (steps, n_features)")
        if not np.isfinite(X).all():
            raise ValueError("sequence contains NaN or Inf features")
        return ((X - self.mean) / self.std).astype(np.float32)

    def channel_outputs(self, sequence, mask=None, exploration: bool = False):
        X = self._standardize_sequence(sequence, mask=mask)
        outputs = np.clip(X @ self.W + self.b, -1.0, 1.0).astype(np.float32)
        if exploration and self.config.exploration_noise > 0.0:
            outputs = np.clip(
                outputs + self.rng.normal(0.0, self.config.exploration_noise, size=outputs.shape),
                -1.0,
                1.0,
            ).astype(np.float32)
        return X, outputs

    def _accumulate_step(self, evidence, active, out):
        evidence = np.asarray(evidence, dtype=np.float32).copy()
        active = np.asarray(active, dtype=bool).copy()
        if active.any():
            gain = float(self.config.evidence_gain)
            evidence[active] = np.clip(evidence[active] + gain * out[active], -1.0, 1.0)
            eliminated = active & (evidence <= float(self.config.elimination_threshold))
            active[eliminated] = False
            evidence[~active] = float(self.config.elimination_threshold)
        return evidence.astype(np.float32), active

    def rollout(self, sequence, label: int, mask=None, train: bool = False, stochastic: bool = False):
        label = int(label)
        X, outputs = self.channel_outputs(sequence, mask=mask, exploration=bool(train and stochastic))
        evidence = np.zeros(10, dtype=np.float32)
        active = np.ones(10, dtype=bool)
        seen_outputs = []
        seen_evidence = []
        decision_step = outputs.shape[0]
        threshold_hit = False
        all_eliminated = False
        action = None

        for step_idx, out in enumerate(outputs, start=1):
            evidence, active = self._accumulate_step(evidence, active, out)
            seen_outputs.append(out)
            seen_evidence.append(evidence.copy())
            decision_step = step_idx
            active_hits = np.flatnonzero(active & (evidence >= float(self.config.decision_threshold)))
            if len(active_hits):
                action = int(active_hits[np.argmax(evidence[active_hits])])
                threshold_hit = True
                break
            if not active.any():
                all_eliminated = True
                break

        probs = _masked_softmax_np(evidence, active, temperature=self.config.temperature)
        if action is None:
            if active.any():
                action = int(self.rng.choice(10, p=probs)) if stochastic else int(np.argmax(np.where(active, evidence, -np.inf)))
            else:
                action = -1
        correct = action == label

        seen_outputs = np.asarray(seen_outputs, dtype=np.float32)
        seen_evidence = np.asarray(seen_evidence, dtype=np.float32)
        if len(seen_outputs):
            correct_signal = seen_outputs[:, label]
            incorrect_outputs = np.delete(seen_outputs, label, axis=1)
            dense_reward = float(np.mean(correct_signal - np.maximum(incorrect_outputs, 0.0).sum(axis=1)))
        else:
            dense_reward = 0.0
        reward = self.config.correct_reward if correct else self.config.wrong_penalty
        reward += self.config.dense_reward_scale * dense_reward
        reward -= self.config.time_penalty * float(decision_step)
        if not threshold_hit:
            reward += self.config.timeout_penalty
        avg_feature = X[:max(1, decision_step)].mean(axis=0).astype(np.float32)
        return {
            "action": int(action),
            "correct": bool(correct),
            "reward": float(reward),
            "dense_reward": dense_reward,
            "steps": int(decision_step),
            "timeout": not threshold_hit,
            "threshold_hit": bool(threshold_hit),
            "all_eliminated": bool(all_eliminated),
            "probs": probs.astype(np.float32),
            "evidence": evidence.astype(np.float32),
            "active": active.astype(bool),
            "outputs": seen_outputs.astype(np.float32),
            "features": X[:len(seen_outputs)].astype(np.float32),
            "evidence_trace": seen_evidence.astype(np.float32),
            "avg_feature": avg_feature,
        }

    def update_batch(self, sequences, labels, mask=None, stochastic: bool = True):
        labels = np.asarray(labels, dtype=np.int64)
        batch_mask = np.ones((len(labels), sequences.shape[1]), dtype=bool) if mask is None else np.asarray(mask, dtype=bool)
        grad_W = np.zeros_like(self.W)
        grad_b = np.zeros_like(self.b)
        rewards, corrects, steps, timeouts, eliminated, dense_rewards, losses, shaping_losses = [], [], [], [], [], [], [], []
        for seq, seq_mask, label in zip(sequences, batch_mask, labels):
            result = self.rollout(seq, int(label), mask=seq_mask, train=True, stochastic=bool(stochastic))
            rewards.append(result["reward"])
            corrects.append(result["correct"])
            steps.append(result["steps"])
            timeouts.append(result["timeout"])
            eliminated.append(result["all_eliminated"])
            dense_rewards.append(result["dense_reward"])
            advantage = float(result["reward"] - self.baseline)
            direction = -result["probs"].astype(np.float32)
            if result["action"] >= 0:
                direction[int(result["action"])] += 1.0
                scale = advantage
                chosen_prob = float(result["probs"][int(result["action"])])
                losses.append(-advantage * np.log(chosen_prob + 1e-8))
            else:
                direction[int(label)] += 1.0
                scale = abs(advantage) if advantage != 0.0 else 1.0
                losses.append(scale)
            grad_W += np.outer(result["avg_feature"], scale * direction)
            grad_b += scale * direction

            shaping_scale = float(getattr(self.config, "channel_shaping_scale", self.config.dense_reward_scale))
            features = result.get("features")
            outputs = result.get("outputs")
            if shaping_scale > 0.0 and features is not None and outputs is not None and len(outputs):
                target = np.full_like(outputs, -0.25, dtype=np.float32)
                target[:, int(label)] = 1.0
                channel_error = (target - outputs).astype(np.float32)
                step_scale = shaping_scale / max(1, outputs.shape[0])
                grad_W += step_scale * (features.T @ channel_error)
                grad_b += shaping_scale * channel_error.mean(axis=0)
                shaping_losses.append(float(np.mean(channel_error * channel_error)))

        scale = 1.0 / max(1, len(labels))
        grad_W *= scale
        grad_b *= scale
        grad_norm = float(np.sqrt(np.sum(grad_W * grad_W) + np.sum(grad_b * grad_b)))
        self.W += self.config.lr * (grad_W - self.config.l2 * self.W)
        self.b += self.config.lr * grad_b
        self.W = np.clip(self.W, -5.0, 5.0).astype(np.float32)
        self.b = np.clip(self.b, -2.0, 2.0).astype(np.float32)
        mean_reward = float(np.mean(rewards))
        decay = float(np.clip(self.config.baseline_decay, 0.0, 0.999))
        self.baseline = decay * self.baseline + (1.0 - decay) * mean_reward
        self.updates += 1
        self.examples_seen += len(labels)
        metrics = {
            "reward": mean_reward,
            "accuracy": float(np.mean(corrects)),
            "steps": float(np.mean(steps)),
            "timeout_rate": float(np.mean(timeouts)),
            "all_eliminated_rate": float(np.mean(eliminated)),
            "dense_reward": float(np.mean(dense_rewards)),
            "policy_loss": float(np.mean(losses)),
            "shaping_loss": float(np.mean(shaping_losses)) if shaping_losses else np.nan,
            "grad_norm": grad_norm,
            "baseline": float(self.baseline),
            "updates": int(self.updates),
            "examples_seen": int(self.examples_seen),
        }
        self.last_metrics = metrics
        return metrics

    def evaluate(self, sequences, labels, mask=None):
        labels = np.asarray(labels, dtype=np.int64)
        batch_mask = np.ones((len(labels), sequences.shape[1]), dtype=bool) if mask is None else np.asarray(mask, dtype=bool)
        results = [self.rollout(seq, int(label), mask=seq_mask, train=False, stochastic=False) for seq, seq_mask, label in zip(sequences, batch_mask, labels)]
        return {
            "reward": float(np.mean([r["reward"] for r in results])),
            "accuracy": float(np.mean([r["correct"] for r in results])),
            "steps": float(np.mean([r["steps"] for r in results])),
            "timeout_rate": float(np.mean([r["timeout"] for r in results])),
            "all_eliminated_rate": float(np.mean([r["all_eliminated"] for r in results])),
        }


def train_evidence_policy(policy, train_sequences, train_labels, train_mask, config: RLMNISTConfig, eval_sequences=None, eval_labels=None, eval_mask=None):
    rng = np.random.default_rng(config.seed)
    train_labels = np.asarray(train_labels, dtype=np.int64)
    history = {"epoch": [], "update": [], "train_accuracy": [], "eval_accuracy": [], "reward": [], "steps": [], "timeout_rate": [], "all_eliminated_rate": [], "policy_loss": []}
    for epoch in range(int(config.epochs)):
        order = rng.permutation(len(train_labels))
        pbar = tqdm(range(0, len(order), int(config.batch_size)), desc=f"evidence RL epoch {epoch + 1}/{config.epochs}")
        last_metrics = None
        for start in pbar:
            idx = order[start:start + int(config.batch_size)]
            last_metrics = policy.update_batch(train_sequences[idx], train_labels[idx], train_mask[idx])
            pbar.set_postfix(acc=f"{last_metrics['accuracy']:.2f}", reward=f"{last_metrics['reward']:.2f}", steps=f"{last_metrics['steps']:.1f}")
        should_eval = eval_sequences is not None and (epoch + 1) % max(1, int(config.eval_every)) == 0
        eval_metrics = policy.evaluate(eval_sequences, eval_labels, eval_mask) if should_eval else {"accuracy": np.nan}
        history["epoch"].append(epoch + 1)
        history["update"].append(policy.updates)
        history["train_accuracy"].append(last_metrics["accuracy"] if last_metrics else np.nan)
        history["eval_accuracy"].append(eval_metrics["accuracy"])
        history["reward"].append(last_metrics["reward"] if last_metrics else np.nan)
        history["steps"].append(last_metrics["steps"] if last_metrics else np.nan)
        history["timeout_rate"].append(last_metrics["timeout_rate"] if last_metrics else np.nan)
        history["all_eliminated_rate"].append(last_metrics["all_eliminated_rate"] if last_metrics else np.nan)
        history["policy_loss"].append(last_metrics["policy_loss"] if last_metrics else np.nan)
    return history


def plot_evidence_training_history(history):
    fig = go.Figure()
    x = history["update"]
    fig.add_trace(go.Scatter(x=x, y=history["train_accuracy"], mode="lines+markers", name="train batch acc", line=dict(color="#4c78a8", width=2)))
    fig.add_trace(go.Scatter(x=x, y=history["eval_accuracy"], mode="lines+markers", name="eval acc", line=dict(color="#f58518", width=2)))
    fig.add_trace(go.Scatter(x=x, y=history["reward"], mode="lines+markers", name="reward", yaxis="y2", line=dict(color="#54a24b", width=2)))
    fig.add_trace(go.Scatter(x=x, y=history["steps"], mode="lines+markers", name="mean steps", yaxis="y3", line=dict(color="#b279a2", width=2, dash="dot")))
    if "all_eliminated_rate" in history:
        fig.add_trace(go.Scatter(x=x, y=history["all_eliminated_rate"], mode="lines+markers", name="all eliminated", line=dict(color="#e45756", width=2, dash="dash")))
    fig.update_layout(
        width=950,
        height=420,
        title="MNIST signed evidence-accumulator RL readout",
        xaxis_title="policy update",
        yaxis=dict(title="accuracy / eliminated", range=[0, 1]),
        yaxis2=dict(title="reward", overlaying="y", side="right", rangemode="tozero"),
        yaxis3=dict(title="steps", overlaying="y", side="right", position=0.94, showgrid=False),
    )
    fig.show()
    return fig


In [ ]:
rl_train_images, rl_train_labels = select_balanced(train_images_all, train_labels_all, RL_MNIST_CONFIG.train_per_digit, seed=31)
rl_test_images, rl_test_labels = select_balanced(test_images_all, test_labels_all, RL_MNIST_CONFIG.test_per_digit, seed=32)
X_rl_train = input_from_images(rl_train_images)
X_rl_test = input_from_images(rl_test_images)

start = time.perf_counter()
rl_train_sequences, rl_train_mask = reservoir.collect_feature_sequences(X_rl_train, batch_size=RL_MNIST_CONFIG.batch_size, progress_desc="RL train sequences")
rl_test_sequences, rl_test_mask = reservoir.collect_feature_sequences(X_rl_test, batch_size=RL_MNIST_CONFIG.batch_size, progress_desc="RL test sequences")
print(f"RL sequence extraction seconds: {time.perf_counter() - start:.1f}")
print("RL sequence shape:", rl_train_sequences.shape, "readout steps:", int(rl_train_mask.sum(axis=1).max()))

evidence_policy = EvidenceAccumulatorPolicy(reservoir.n_features, RL_MNIST_CONFIG)
evidence_policy.set_normalizer(rl_train_sequences, rl_train_mask)
rl_history = train_evidence_policy(
    evidence_policy,
    rl_train_sequences,
    rl_train_labels,
    rl_train_mask,
    RL_MNIST_CONFIG,
    eval_sequences=rl_test_sequences,
    eval_labels=rl_test_labels,
    eval_mask=rl_test_mask,
)
rl_eval_metrics = evidence_policy.evaluate(rl_test_sequences, rl_test_labels, rl_test_mask)
print("final RL eval:", rl_eval_metrics)
rl_training_fig = plot_evidence_training_history(rl_history)

In [ ]:
try:
    import threading
    import ipywidgets as widgets
    from IPython.display import display
except Exception as exc:
    widgets = None
    print(f"ipywidgets unavailable: {exc}")


class LiveMNISTSpikeEngineView:
    def __init__(self, reservoir: SpikeEngineMNISTReservoir, probe, history=None):
        if widgets is None:
            raise RuntimeError("ipywidgets is required for the live view")
        self.reservoir = reservoir
        self.probe = probe
        self.rng = np.random.default_rng(123)
        self._stop = threading.Event()
        self._train_stop = threading.Event()
        self._thread = None
        self._train_thread = None
        self._lock = threading.RLock()
        self.current_digit = None
        self.current_image = np.zeros((28, 28), dtype=np.uint8)
        self.current_input = np.zeros(self.reservoir.n_input, dtype=np.float32)
        self.readout_sum = np.zeros(self.reservoir.n_features, dtype=np.float32)
        self.readout_count = 0
        
        self.sequence_step = 0
        self.last_eval_acc = None
        self.last_batch_acc = getattr(self.probe, "last_batch_acc", None)
        base_history = history or {"updates": [], "batch_acc": [], "eval_acc": [], "loss": []}
        self.history = {key: list(base_history.get(key, [])) for key in ["updates", "batch_acc", "eval_acc", "loss"]}

        self.input_fig = go.FigureWidget(data=[go.Heatmap(z=np.zeros(self.reservoir.config.input_shape), colorscale="Gray", zmin=0, zmax=1, showscale=False)])
        self.input_fig.update_layout(width=310, height=310, margin=dict(l=0, r=0, b=0, t=24), title=dict(text="14x14 input", x=0.5, y=0.98, font=dict(size=13)), xaxis=dict(visible=False, fixedrange=True), yaxis=dict(visible=False, fixedrange=True, autorange="reversed"))
        self.reservoir_fig = go.FigureWidget(data=[go.Heatmap(z=np.zeros((self.reservoir.config.reservoir_side, self.reservoir.config.reservoir_side)), colorscale="Viridis", showscale=False)])
        self.reservoir_fig.update_layout(width=430, height=430, margin=dict(l=0, r=0, b=0, t=24), title=dict(text="SpikeEngine membrane", x=0.5, y=0.98, font=dict(size=13)), xaxis=dict(visible=False, fixedrange=True), yaxis=dict(visible=False, fixedrange=True, autorange="reversed"))
        self.probe_fig = go.FigureWidget(data=[go.Bar(x=list(range(10)), y=np.full(10, 0.1, dtype=np.float32), marker_color="#4c78a8")])
        self.probe_fig.update_layout(width=640, height=260, margin=dict(l=30, r=10, b=30, t=24), title=dict(text="readout probabilities", x=0.5, y=0.98, font=dict(size=13)), yaxis=dict(range=[0, 1], fixedrange=True), xaxis=dict(dtick=1, fixedrange=True))
        self.train_fig = go.FigureWidget(data=[
            go.Scatter(x=[], y=[], mode="lines+markers", name="batch acc", line=dict(color="#4c78a8", width=2)),
            go.Scatter(x=[], y=[], mode="lines+markers", name="eval acc", line=dict(color="#f58518", width=2)),
            go.Scatter(x=[], y=[], mode="lines", name="loss", yaxis="y2", line=dict(color="#54a24b", width=2, dash="dot")),
        ])
        self.train_fig.update_layout(width=950, height=300, margin=dict(l=45, r=45, t=30, b=40), title=dict(text="live readout training", x=0.5, font=dict(size=14)), xaxis_title="readout update", yaxis=dict(title="accuracy", range=[0, 1], fixedrange=True), yaxis2=dict(title="loss", overlaying="y", side="right", rangemode="tozero"))

        self.digit_buttons = [widgets.Button(description=str(d), layout=widgets.Layout(width="42px")) for d in range(10)]
        for digit, button in enumerate(self.digit_buttons):
            button.on_click(lambda _, d=digit: self.present_digit(d))
        self.train_batch_button = widgets.Button(description="Train Batch", icon="graduation-cap", button_style="info")
        self.auto_train_button = widgets.Button(description="Auto Train", icon="play", button_style="success")
        self.stop_button = widgets.Button(description="Stop", icon="stop", button_style="danger")
        self.reset_state_button = widgets.Button(description="Reset Engine", icon="refresh")
        self.reset_probe_button = widgets.Button(description="Reset Probe", icon="eraser")
        self.eval_button = widgets.Button(description="Eval", icon="check")
        self.apply_dynamics_button = widgets.Button(description="Apply Dynamics", icon="sliders", button_style="warning")
        self.train_batch_button.on_click(lambda _: self.train_one_batch_async())
        self.auto_train_button.on_click(lambda _: self.start_auto_train())
        self.stop_button.on_click(lambda _: self.stop_all())
        self.reset_state_button.on_click(lambda _: self.reset_state())
        self.reset_probe_button.on_click(lambda _: self.reset_probe())
        self.eval_button.on_click(lambda _: self.evaluate_async())
        self.apply_dynamics_button.on_click(lambda _: self.apply_dynamics())

        self.sample_source = widgets.Dropdown(options=["train", "test"], value="train", description="source")
        self.train_on_present = widgets.Checkbox(value=True, description="train on presented")
        self.reset_before_image = widgets.Checkbox(value=True, description="reset before image")
        self.reset_on_dynamics = widgets.Checkbox(value=True, description="reset on dynamics apply", style={"description_width": "initial"})
        cfg = self.reservoir.config
        slider_style = {"description_width": "initial"}
        self.input_gain = widgets.FloatSlider(value=cfg.input_gain, min=0.0, max=6.0, step=0.05, description="input gain", continuous_update=False, style=slider_style)
        self.recurrent_scale = widgets.FloatSlider(value=cfg.recurrent_scale, min=0.0, max=2.5, step=0.01, readout_format=".2f", description="recurrent scale", continuous_update=False, style=slider_style)
        self.decay_rate = widgets.FloatSlider(value=cfg.decay_rate, min=0.0, max=0.75, step=0.01, readout_format=".2f", description="decay", continuous_update=False, style=slider_style)
        self.spike_threshold = widgets.FloatSlider(value=cfg.spike_threshold, min=0.05, max=3.0, step=0.05, readout_format=".2f", description="threshold", continuous_update=False, style=slider_style)
        self.resting_mp = widgets.FloatSlider(value=cfg.resting_mp, min=-0.5, max=0.95, step=0.01, readout_format=".2f", description="resting MP", continuous_update=False, style=slider_style)
        self.spike_tau = widgets.FloatSlider(value=cfg.spike_tau, min=1.0, max=80.0, step=1.0, readout_format=".0f", description="spike trace tau", continuous_update=False, style=slider_style)
        self.feature_voltage_scale = widgets.FloatLogSlider(value=cfg.feature_voltage_scale, base=10, min=-2, max=1, step=0.1, description="voltage scale", continuous_update=False, style=slider_style)
        self.pre_steps = widgets.IntSlider(value=cfg.pre_steps, min=0, max=50, description="pre", continuous_update=False)
        self.on_steps = widgets.IntSlider(value=cfg.on_steps, min=1, max=120, description="on", continuous_update=False)
        self.off_steps = widgets.IntSlider(value=cfg.off_steps, min=0, max=80, description="off", continuous_update=False)
        self.readout_start = widgets.IntSlider(value=cfg.readout_start, min=0, max=180, description="readout start", continuous_update=False, style=slider_style)
        self.readout_stride = widgets.IntSlider(value=cfg.readout_stride, min=1, max=20, description="readout stride", continuous_update=False, style=slider_style)
        self.steps_per_frame = widgets.IntSlider(value=1, min=1, max=20, description="steps/frame", continuous_update=False, style=slider_style)
        self.delay_ms = widgets.IntSlider(value=20, min=0, max=250, step=5, description="delay ms", continuous_update=False, style=slider_style)
        self.reservoir_zmax = widgets.FloatSlider(value=1.0, min=0.05, max=5.0, step=0.05, description="mp zmax", continuous_update=False, style=slider_style)
        self.train_batch_size = widgets.IntSlider(value=64, min=1, max=2048, step=1, description="batch", continuous_update=False)
        self.auto_delay_ms = widgets.IntSlider(value=100, min=0, max=2000, step=25, description="train delay", continuous_update=False, style=slider_style)
        self.eval_per_digit = widgets.IntSlider(value=20, min=5, max=200, step=5, description="eval/digit", continuous_update=False, style=slider_style)
        self.history_points = widgets.IntSlider(value=1000, min=50, max=10000, step=50, description="plot points", continuous_update=False, style=slider_style)
        self.rls_delta = widgets.FloatLogSlider(value=float(getattr(self.probe, "delta", PROBE_CONFIG.rls_delta)), base=10, min=-6, max=2, step=0.25, description="RLS delta", continuous_update=False, style=slider_style)
        self.rls_forgetting = widgets.FloatSlider(value=float(getattr(self.probe, "forgetting", PROBE_CONFIG.rls_forgetting)), min=0.90, max=1.0, step=0.001, readout_format=".3f", description="RLS forget", continuous_update=False, style=slider_style)
        self.online_lr = widgets.FloatLogSlider(value=float(getattr(self.probe, "lr", PROBE_CONFIG.online_lr)), base=10, min=-6, max=0, step=0.1, description="softmax lr", continuous_update=False, style=slider_style)
        self.online_l2 = widgets.FloatLogSlider(value=float(getattr(self.probe, "l2", PROBE_CONFIG.online_l2)), base=10, min=-8, max=1, step=0.25, description="softmax L2", continuous_update=False, style=slider_style)
        if not hasattr(self.probe, "delta"):
            self.rls_delta.layout.display = "none"
        if not hasattr(self.probe, "forgetting"):
            self.rls_forgetting.layout.display = "none"
        if not hasattr(self.probe, "lr"):
            self.online_lr.layout.display = "none"
        if not hasattr(self.probe, "l2"):
            self.online_l2.layout.display = "none"
        self.status = widgets.HTML(value="idle")
        self._refresh_training_figure()

    def _sync_probe_hyperparams(self):
        if hasattr(self.probe, "delta"):
            self.probe.delta = float(self.rls_delta.value)
        if hasattr(self.probe, "forgetting"):
            self.probe.forgetting = float(self.rls_forgetting.value)
        if hasattr(self.probe, "lr"):
            self.probe.lr = float(self.online_lr.value)
        if hasattr(self.probe, "l2"):
            self.probe.l2 = float(self.online_l2.value)

    def _sync_config(self, reset_state: bool = False):
        cfg = self.reservoir.config
        self.reservoir.apply_runtime_controls(
            input_gain=float(self.input_gain.value),
            recurrent_scale=float(self.recurrent_scale.value),
            decay_rate=float(self.decay_rate.value),
            spike_threshold=float(self.spike_threshold.value),
            resting_mp=float(self.resting_mp.value),
            spike_tau=float(self.spike_tau.value),
            feature_voltage_scale=float(self.feature_voltage_scale.value),
            reset_state=reset_state,
        )
        cfg.pre_steps = int(self.pre_steps.value)
        cfg.on_steps = int(self.on_steps.value)
        cfg.off_steps = int(self.off_steps.value)
        cfg.readout_start = int(self.readout_start.value)
        cfg.readout_stride = int(self.readout_stride.value)
        self._sync_probe_hyperparams()

    def _history_limit(self):
        widget = getattr(self, "history_points", None)
        return max(1, int(widget.value if widget is not None else 1000))

    def _trim_history(self):
        limit = self._history_limit()
        for key in ["updates", "batch_acc", "eval_acc", "loss"]:
            values = list(self.history.get(key, []))
            if len(values) > limit:
                self.history[key] = values[-limit:]

    def _history_array(self, key):
        x_len = len(self.history.get("updates", []))
        if x_len <= 0:
            return np.asarray([], dtype=np.float32)
        values = list(self.history.get(key, []))[-x_len:]
        if len(values) < x_len:
            values = [np.nan] * (x_len - len(values)) + values
        return np.asarray(values, dtype=np.float32)

    def _record_training_point(self, *, batch_acc=None, eval_acc=None, loss=None):
        update = int(getattr(self.probe, "updates", len(self.history["updates"])))
        self.history["updates"].append(update)
        self.history["batch_acc"].append(np.nan if batch_acc is None else float(batch_acc))
        self.history["eval_acc"].append(np.nan if eval_acc is None else float(eval_acc))
        self.history["loss"].append(np.nan if loss is None else float(loss))
        self._trim_history()
        self._refresh_training_figure()

    def _refresh_training_figure(self):
        self._trim_history()
        x = np.asarray(self.history.get("updates", []), dtype=np.float32)
        with self.train_fig.batch_update():
            self.train_fig.data[0].x = x
            self.train_fig.data[0].y = self._history_array("batch_acc")
            self.train_fig.data[1].x = x
            self.train_fig.data[1].y = self._history_array("eval_acc")
            self.train_fig.data[2].x = x
            self.train_fig.data[2].y = self._history_array("loss")

    def _sample_digit(self, digit: int):
        images, labels = (train_images_all, train_labels_all) if self.sample_source.value == "train" else (test_images_all, test_labels_all)
        choices = np.flatnonzero(labels == digit)
        idx = int(self.rng.choice(choices))
        image = images[idx]
        return image, input_from_images(image[None, :, :])[0]

    def _total_steps(self):
        return int(self.pre_steps.value + self.on_steps.value + self.off_steps.value)

    def _current_feature(self):
        if self.readout_count > 0:
            return self.readout_sum / self.readout_count
        return self.reservoir._snapshot_feature()

    def _draw(self, env_value=0.0):
        feature = self._current_feature()
        probs = self.probe.probabilities(feature[None, :])[0]
        pred = int(np.argmax(probs))
        input_grid = (self.current_input * env_value).reshape(self.reservoir.config.input_shape)
        reservoir_grid = self.reservoir.current_frame()
        last = cp.asnumpy(self.reservoir.engine.last_spiked).astype(np.int32)
        recent = float(np.mean((self.reservoir.tick - last) <= 1))
        ever = float(np.mean(last > 0))
        with self.input_fig.batch_update():
            self.input_fig.data[0].z = input_grid
        with self.reservoir_fig.batch_update():
            self.reservoir_fig.data[0].z = reservoir_grid
            self.reservoir_fig.data[0].zmax = float(self.reservoir_zmax.value)
        with self.probe_fig.batch_update():
            self.probe_fig.data[0].y = probs
            colors = ["#4c78a8"] * 10
            colors[pred] = "#f58518"
            self.probe_fig.data[0].marker.color = colors
        true_text = "?" if self.current_digit is None else str(self.current_digit)
        eval_text = "n/a" if self.last_eval_acc is None else f"{self.last_eval_acc:.3f}"
        batch_text = "n/a" if getattr(self.probe, "last_batch_acc", None) is None else f"{self.probe.last_batch_acc:.3f}"
        loss_text = "n/a" if getattr(self.probe, "last_loss", None) is None else f"{self.probe.last_loss:.4f}"
        self.status.value = (
            f"true={true_text} pred={pred} p={probs[pred]:.3f} step={self.sequence_step}/{self._total_steps()} "
            f"readout_n={self.readout_count} recent={recent:.3f} ever={ever:.3f} "
            f"updates={getattr(self.probe, 'updates', 0)} seen={getattr(self.probe, 'examples_seen', 0)} "
            f"loss={loss_text} batch_acc={batch_text} eval={eval_text} "
            f"mp_mean={reservoir_grid.mean():.3f} mp_max={reservoir_grid.max():.3f}"
        )

    def apply_dynamics(self):
        self.stop_presentation()
        with self._lock:
            self._sync_config(reset_state=bool(self.reset_on_dynamics.value))
            self.readout_sum.fill(0.0)
            self.readout_count = 0
            self.sequence_step = 0
            self._draw(0.0)
            weight_stats = self.reservoir.weight_summary.get("weight_stats", {}).get("after", {})
            self.status.value += f" | w_rms={float(weight_stats.get('rms', np.nan)):.4f}"

    def _run_sequence(self):
        self._sync_config()
        self.readout_sum.fill(0.0)
        self.readout_count = 0
        self.sequence_step = 0
        total = self._total_steps()
        while not self._stop.is_set() and self.sequence_step < total:
            with self._lock:
                env_value = 0.0
                for _ in range(int(self.steps_per_frame.value)):
                    if self.sequence_step >= total:
                        break
                    env_value = self.reservoir.envelope(self.sequence_step)
                    self.reservoir.step(self.current_input, sequence_step=self.sequence_step)
                    if self.sequence_step >= self.reservoir.config.readout_start and (self.sequence_step - self.reservoir.config.readout_start) % self.reservoir.config.readout_stride == 0:
                        self.readout_sum += self.reservoir._snapshot_feature()
                        self.readout_count += 1
                    self.sequence_step += 1
                self._draw(env_value)
            delay = int(self.delay_ms.value) / 1000.0
            if delay:
                time.sleep(delay)
        with self._lock:
            if self.current_digit is not None and self.train_on_present.value and self.readout_count > 0:
                self._sync_probe_hyperparams()
                loss, batch_acc = self.probe.partial_fit(self._current_feature()[None, :], [self.current_digit])
                self.last_batch_acc = batch_acc
                self._record_training_point(batch_acc=batch_acc, loss=loss)
            self._draw(0.0)

    def present_digit(self, digit: int):
        self.stop_presentation()
        with self._lock:
            self.current_digit = int(digit)
            self.current_image, self.current_input = self._sample_digit(digit)
            self._sync_config(reset_state=bool(self.reset_before_image.value))
        self._stop.clear()
        self._thread = threading.Thread(target=self._run_sequence, daemon=True, name="mnist-spikeengine-present")
        self._thread.start()

    def _training_features_from_random_batch(self):
        self._sync_config()
        images, labels = sample_random_labeled(train_images_all, train_labels_all, int(self.train_batch_size.value), self.rng)
        inputs = input_from_images(images)
        features = self.reservoir.collect_features(inputs, batch_size=min(128, int(self.train_batch_size.value)), progress_desc=None)
        return features, labels

    def train_one_batch(self):
        with self._lock:
            self._sync_probe_hyperparams()
            features, labels = self._training_features_from_random_batch()
            loss, batch_acc = self.probe.partial_fit(features, labels)
            self.last_batch_acc = batch_acc
            self._record_training_point(batch_acc=batch_acc, loss=loss)
            self._draw(0.0)

    def train_one_batch_async(self):
        threading.Thread(target=self.train_one_batch, daemon=True, name="mnist-spikeengine-train-batch").start()

    def _auto_train_loop(self):
        while not self._train_stop.is_set():
            self.train_one_batch()
            delay = int(self.auto_delay_ms.value) / 1000.0
            if delay:
                time.sleep(delay)

    def start_auto_train(self):
        if self._train_thread is not None and self._train_thread.is_alive():
            return
        self._train_stop.clear()
        self._train_thread = threading.Thread(target=self._auto_train_loop, daemon=True, name="mnist-spikeengine-auto-train")
        self._train_thread.start()

    def evaluate(self):
        with self._lock:
            self._sync_config()
            images, labels = sample_random_labeled(test_images_all, test_labels_all, int(self.eval_per_digit.value) * 10, self.rng)
            features = self.reservoir.collect_features(input_from_images(images), batch_size=128, progress_desc=None)
            self.last_eval_acc = accuracy(self.probe, features, labels)
            self._record_training_point(eval_acc=self.last_eval_acc, loss=getattr(self.probe, "last_loss", None))
            self._draw(0.0)

    def evaluate_async(self):
        threading.Thread(target=self.evaluate, daemon=True, name="mnist-spikeengine-eval").start()

    def stop_presentation(self):
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=2)

    def stop_training(self):
        self._train_stop.set()
        if self._train_thread is not None:
            self._train_thread.join(timeout=2)

    def stop_all(self):
        self.stop_presentation()
        self.stop_training()

    def reset_state(self):
        self.stop_presentation()
        with self._lock:
            self.reservoir.reset()
            self.current_digit = None
            self.current_input.fill(0.0)
            self.readout_sum.fill(0.0)
            self.readout_count = 0
            self.sequence_step = 0
            self._draw(0.0)

    def reset_probe(self):
        with self._lock:
            self._sync_probe_hyperparams()
            if hasattr(self.probe, "reset_weights"):
                if hasattr(self.probe, "delta"):
                    self.probe.reset_weights(delta=float(self.rls_delta.value))
                else:
                    self.probe.reset_weights()
            self.last_eval_acc = None
            self.last_batch_acc = None
            self.history = {"updates": [], "batch_acc": [], "eval_acc": [], "loss": []}
            self._refresh_training_figure()
            self._draw(0.0)

    def display(self):
        controls = widgets.VBox([
            widgets.HBox(self.digit_buttons + [self.train_batch_button, self.auto_train_button, self.stop_button]),
            widgets.HBox([self.reset_state_button, self.reset_probe_button, self.eval_button, self.apply_dynamics_button, self.sample_source, self.train_on_present, self.reset_before_image, self.reset_on_dynamics]),
            widgets.HBox([self.input_gain, self.recurrent_scale, self.decay_rate, self.spike_threshold]),
            widgets.HBox([self.resting_mp, self.spike_tau, self.feature_voltage_scale, self.pre_steps, self.on_steps, self.off_steps]),
            widgets.HBox([self.readout_start, self.readout_stride, self.steps_per_frame, self.delay_ms, self.reservoir_zmax]),
            widgets.HBox([self.train_batch_size, self.rls_delta, self.rls_forgetting, self.online_lr, self.online_l2]),
            widgets.HBox([self.auto_delay_ms, self.eval_per_digit, self.history_points]),
            self.status,
        ])
        display(widgets.VBox([controls, self.train_fig, widgets.HBox([self.input_fig, self.reservoir_fig]), self.probe_fig]))


try:
    live_mnist.stop_all()
except NameError:
    pass

live_mnist = LiveMNISTSpikeEngineView(reservoir, rls_probe, history=mnist_training_history)
live_mnist.display()

## Evidence Accumulator Workbench

This is the preferred RL-style MNIST workflow. It presents digits through the SpikeEngine reservoir, accumulates ten signed evidence channels from zero with `[-1, 1]` bounds, chooses the first channel to reach `+1`, eliminates any channel that reaches `-1`, and trains from reward using correctness, wrong-answer penalty, timeout penalty, time cost, and dense signed-drift shaping.


In [ ]:
try:
    import copy
    import threading
    import traceback
    import ipywidgets as widgets
    from IPython.display import display
except Exception as exc:
    widgets = None
    print(f"ipywidgets unavailable: {exc}")


EVIDENCE_HISTORY_KEYS = ["updates", "train_accuracy", "eval_accuracy", "reward", "steps", "timeout_rate", "all_eliminated_rate", "policy_loss"]


def _copy_evidence_history(history=None):
    base = history if isinstance(history, dict) else {}
    copied = {}
    for key in EVIDENCE_HISTORY_KEYS:
        source_key = "update" if key == "updates" and "update" in base else key
        copied[key] = list(base.get(source_key, []))
    return copied


def _history_with_policy_tail(history, policy):
    copied = _copy_evidence_history(history)
    metrics = dict(getattr(policy, "last_metrics", {}) or {})
    updates = int(getattr(policy, "updates", 0))
    if updates > 0 and (not copied["updates"] or int(copied["updates"][-1]) != updates):
        copied["updates"].append(updates)
        copied["train_accuracy"].append(float(metrics.get("accuracy", np.nan)))
        copied["eval_accuracy"].append(np.nan)
        copied["reward"].append(float(metrics.get("reward", np.nan)))
        copied["steps"].append(float(metrics.get("steps", np.nan)))
        copied["timeout_rate"].append(float(metrics.get("timeout_rate", np.nan)))
        copied["all_eliminated_rate"].append(float(metrics.get("all_eliminated_rate", np.nan)))
        copied["policy_loss"].append(float(metrics.get("policy_loss", np.nan)))
    return copied


def sample_balanced_random_labeled(images, labels, count: int, rng):
    count = int(count)
    if count <= 0:
        return images[:0], labels[:0]
    if count < 10:
        return sample_random_labeled(images, labels, count, rng)
    labels = np.asarray(labels)
    per_digit = count // 10
    remainder = count % 10
    digits = np.arange(10)
    extra_digits = set(rng.choice(digits, size=remainder, replace=False).tolist()) if remainder else set()
    indices = []
    for digit in digits:
        take = per_digit + (1 if int(digit) in extra_digits else 0)
        choices = np.flatnonzero(labels == digit)
        indices.extend(rng.choice(choices, size=take, replace=take > len(choices)).tolist())
    rng.shuffle(indices)
    return images[indices], labels[indices]


def _make_default_evidence_policy():
    policy = EvidenceAccumulatorPolicy(reservoir.n_features, RL_MNIST_CONFIG)
    if "rl_train_sequences" in globals() and "rl_train_mask" in globals():
        policy.set_normalizer(rl_train_sequences, rl_train_mask)
    elif "train_features" in globals():
        policy.set_normalizer(train_features)
    return policy


class EvidenceAccumulatorMNISTWorkbench:
    def __init__(self, reservoir: SpikeEngineMNISTReservoir, policy: EvidenceAccumulatorPolicy, config: RLMNISTConfig = RL_MNIST_CONFIG, history=None):
        if widgets is None:
            raise RuntimeError("ipywidgets is required for the workbench")
        self.reservoir = reservoir
        self.policy = policy
        active_config = getattr(policy, "config", None)
        self.config = replace(active_config if active_config is not None else config)
        self.policy.config = replace(self.config)
        self.rng = np.random.default_rng(self.config.seed + 101)
        self._stop = threading.Event()
        self._train_stop = threading.Event()
        self._thread = None
        self._train_thread = None
        self._lock = threading.RLock()
        self.current_digit = None
        self.current_input = np.zeros(self.reservoir.n_input, dtype=np.float32)
        self.current_evidence = np.zeros(10, dtype=np.float32)
        self.current_active = np.ones(10, dtype=bool)
        self.current_outputs = np.zeros(10, dtype=np.float32)
        self.current_evidence_trace = [self.current_evidence.copy()]
        self.current_sequence = []
        self.feature_mask = None
        self.sequence_step = 0
        self.readout_count = 0
        self.last_eval = None
        self.last_rollout = None
        self.last_train_monitor = None
        self.last_guard_status = ""
        self.handoff_eval_inputs = None
        self.handoff_eval_labels = None
        self.handoff_eval_sequences = None
        self.handoff_eval_mask = None
        self.handoff_eval_metrics = None
        self.handoff_trial = None
        self.handoff_signature = None
        self.channel_colors = ["#4c78a8", "#f58518", "#54a24b", "#e45756", "#72b7b2", "#b279a2", "#ff9da6", "#9d755d", "#bab0ac", "#79706e"]
        self.history = _history_with_policy_tail(history, self.policy)

        self.input_fig = go.FigureWidget(data=[go.Heatmap(z=np.zeros(self.reservoir.config.input_shape), colorscale="Gray", zmin=0, zmax=1, showscale=False)])
        self.input_fig.update_layout(width=310, height=310, margin=dict(l=0, r=0, b=0, t=24), title=dict(text="14x14 input", x=0.5, y=0.98, font=dict(size=13)), xaxis=dict(visible=False, fixedrange=True), yaxis=dict(visible=False, fixedrange=True, autorange="reversed"))
        self.reservoir_fig = go.FigureWidget(data=[go.Heatmap(z=np.zeros((self.reservoir.config.reservoir_side, self.reservoir.config.reservoir_side)), colorscale="Viridis", showscale=False)])
        self.reservoir_fig.update_layout(width=430, height=430, margin=dict(l=0, r=0, b=0, t=24), title=dict(text="SpikeEngine membrane", x=0.5, y=0.98, font=dict(size=13)), xaxis=dict(visible=False, fixedrange=True), yaxis=dict(visible=False, fixedrange=True, autorange="reversed"))
        self.evidence_fig = go.FigureWidget(data=[
            go.Bar(x=list(range(10)), y=np.zeros(10, dtype=np.float32), name="evidence", marker_color="#4c78a8"),
            go.Scatter(x=[-0.5, 9.5], y=[self.config.decision_threshold, self.config.decision_threshold], mode="lines", name="choose", line=dict(color="#54a24b", width=2, dash="dash")),
            go.Scatter(x=[-0.5, 9.5], y=[self.config.elimination_threshold, self.config.elimination_threshold], mode="lines", name="eliminate", line=dict(color="#e45756", width=2, dash="dash")),
        ])
        self.evidence_fig.update_layout(width=640, height=300, margin=dict(l=35, r=15, b=35, t=28), title=dict(text="signed evidence channels", x=0.5, font=dict(size=13)), yaxis=dict(range=[-1.05, 1.05], fixedrange=True), xaxis=dict(dtick=1, fixedrange=True))
        self.output_fig = go.FigureWidget(data=[go.Bar(x=list(range(10)), y=np.zeros(10, dtype=np.float32), marker_color="#72b7b2")])
        self.output_fig.update_layout(width=640, height=220, margin=dict(l=35, r=15, b=35, t=28), title=dict(text="zero-centered channel drift", x=0.5, font=dict(size=13)), yaxis=dict(range=[-1, 1], fixedrange=True), xaxis=dict(dtick=1, fixedrange=True))
        self.accumulation_fig = go.FigureWidget(data=[
            *[go.Scatter(x=[0], y=[0.0], mode="lines", name=str(d), line=dict(color=self.channel_colors[d], width=1.8)) for d in range(10)],
            go.Scatter(x=[0, 1], y=[self.config.decision_threshold, self.config.decision_threshold], mode="lines", name="choose", line=dict(color="#54a24b", width=2, dash="dash")),
            go.Scatter(x=[0, 1], y=[self.config.elimination_threshold, self.config.elimination_threshold], mode="lines", name="eliminate", line=dict(color="#e45756", width=2, dash="dash")),
        ])
        self.accumulation_fig.update_layout(width=520, height=430, margin=dict(l=45, r=15, b=40, t=28), title=dict(text="accumulation course", x=0.5, font=dict(size=13)), xaxis=dict(title="readout", rangemode="tozero", fixedrange=True), yaxis=dict(range=[-1.05, 1.05], fixedrange=True))
        self.train_fig = go.FigureWidget(data=[
            go.Scatter(x=[], y=[], mode="lines+markers", name="train acc", line=dict(color="#4c78a8", width=2)),
            go.Scatter(x=[], y=[], mode="lines+markers", name="eval acc", line=dict(color="#f58518", width=2)),
            go.Scatter(x=[], y=[], mode="lines", name="reward", yaxis="y2", line=dict(color="#54a24b", width=2)),
            go.Scatter(x=[], y=[], mode="lines", name="steps", yaxis="y2", line=dict(color="#b279a2", width=2, dash="dot")),
            go.Scatter(x=[], y=[], mode="lines", name="timeout", line=dict(color="#e45756", width=2, dash="dash")),
            go.Scatter(x=[], y=[], mode="lines", name="all eliminated", line=dict(color="#79706e", width=2, dash="dot")),
        ])
        self.train_fig.update_layout(width=980, height=340, margin=dict(l=45, r=70, t=32, b=40), title=dict(text="evidence policy training", x=0.5, font=dict(size=14)), xaxis_title="policy update", yaxis=dict(title="accuracy", range=[0, 1]), yaxis2=dict(title="reward / steps", overlaying="y", side="right", rangemode="tozero"))

        self.digit_buttons = [widgets.Button(description=str(d), layout=widgets.Layout(width="42px")) for d in range(10)]
        for digit, button in enumerate(self.digit_buttons):
            button.on_click(lambda _, d=digit: self.present_digit(d))
        self.train_button = widgets.Button(description="Train Batch", icon="graduation-cap", button_style="info")
        self.train_many_button = widgets.Button(description="Train N", icon="repeat", button_style="success")
        self.auto_train_button = widgets.Button(description="Auto Train", icon="play", button_style="success")
        self.eval_button = widgets.Button(description="Evaluate", icon="check", button_style="primary")
        self.stop_button = widgets.Button(description="Stop", icon="stop", button_style="danger")
        self.reset_engine_button = widgets.Button(description="Reset Engine", icon="refresh")
        self.reset_policy_button = widgets.Button(description="Reset Policy", icon="eraser")
        self.apply_button = widgets.Button(description="Apply Params", icon="sliders", button_style="warning")
        self.train_button.on_click(lambda _: self.train_one_batch_async())
        self.train_many_button.on_click(lambda _: self.train_n_batches_async())
        self.auto_train_button.on_click(lambda _: self.start_auto_train())
        self.eval_button.on_click(lambda _: self.evaluate_async())
        self.stop_button.on_click(lambda _: self.stop_all())
        self.reset_engine_button.on_click(lambda _: self.reset_engine())
        self.reset_policy_button.on_click(lambda _: self.reset_policy())
        self.apply_button.on_click(lambda _: self.apply_params())

        slider_style = {"description_width": "initial"}
        rcfg = self.reservoir.config
        cfg = self.config
        self.sample_source = widgets.Dropdown(options=["train", "test"], value="train", description="source")
        self.train_on_present = widgets.Checkbox(value=False, description="train on presented")
        self.balanced_batches = widgets.Checkbox(value=True, description="balanced batches", style=slider_style)
        self.exclude_input_readout = widgets.Checkbox(value=bool(getattr(cfg, "readout_exclude_input", True)), description="mask input readout", style=slider_style)
        self.neighborhood_hops = widgets.IntSlider(value=int(getattr(cfg, "readout_neighborhood_hops", 0)), min=0, max=3, step=1, description="mask hops", continuous_update=False, style=slider_style)
        self.reset_before_digit = widgets.Checkbox(value=True, description="reset before digit", style=slider_style)
        self.stochastic_present = widgets.Checkbox(value=False, description="stochastic present", style=slider_style)
        self.train_batch_size = widgets.IntSlider(value=cfg.batch_size, min=1, max=1024, step=1, description="batch", continuous_update=False)
        self.train_batches = widgets.IntSlider(value=5, min=1, max=200, step=1, description="N batches", continuous_update=False, style=slider_style)
        self.eval_per_digit = widgets.IntSlider(value=cfg.test_per_digit, min=1, max=200, step=1, description="eval/digit", continuous_update=False, style=slider_style)
        self.history_points = widgets.IntSlider(value=1000, min=50, max=10000, step=50, description="plot points", continuous_update=False, style=slider_style)
        self.eval_every_batches = widgets.IntSlider(value=0, min=0, max=200, step=1, description="eval every", continuous_update=False, style=slider_style)
        self.train_stochastic = widgets.Checkbox(value=True, description="stochastic updates", style=slider_style)
        self.guard_updates = widgets.Checkbox(value=True, description="guard optuna", style=slider_style)
        self.guard_acc_drop = widgets.FloatSlider(value=0.02, min=0.0, max=0.50, step=0.01, readout_format=".2f", description="guard acc drop", continuous_update=False, style=slider_style)
        self.guard_reward_drop = widgets.FloatSlider(value=0.10, min=0.0, max=2.0, step=0.05, readout_format=".2f", description="guard reward drop", continuous_update=False, style=slider_style)
        self.auto_delay_ms = widgets.IntSlider(value=100, min=0, max=2000, step=25, description="train delay", continuous_update=False, style=slider_style)
        self.delay_ms = widgets.IntSlider(value=30, min=0, max=500, step=5, description="present delay", continuous_update=False, style=slider_style)
        self.steps_per_frame = widgets.IntSlider(value=1, min=1, max=20, step=1, description="steps/frame", continuous_update=False, style=slider_style)

        self.lr = widgets.FloatLogSlider(value=cfg.lr, base=10, min=-5, max=0, step=0.1, description="policy lr", continuous_update=False, style=slider_style)
        self.l2 = widgets.FloatLogSlider(value=cfg.l2, base=10, min=-8, max=-1, step=0.25, description="policy L2", continuous_update=False, style=slider_style)
        self.threshold = widgets.FloatSlider(value=cfg.decision_threshold, min=0.10, max=1.0, step=0.05, readout_format=".2f", description="+threshold", continuous_update=False, style=slider_style)
        self.reject_threshold = widgets.FloatSlider(value=cfg.elimination_threshold, min=-1.0, max=-0.10, step=0.05, readout_format=".2f", description="-threshold", continuous_update=False, style=slider_style)
        self.evidence_gain = widgets.FloatSlider(value=cfg.evidence_gain, min=0.01, max=2.0, step=0.01, readout_format=".2f", description="evidence gain", continuous_update=False, style=slider_style)
        self.temperature = widgets.FloatLogSlider(value=cfg.temperature, base=10, min=-2, max=1, step=0.1, description="temperature", continuous_update=False, style=slider_style)
        self.exploration_noise = widgets.FloatSlider(value=cfg.exploration_noise, min=0.0, max=0.5, step=0.01, readout_format=".2f", description="explore noise", continuous_update=False, style=slider_style)
        self.correct_reward = widgets.FloatSlider(value=cfg.correct_reward, min=0.0, max=5.0, step=0.05, readout_format=".2f", description="correct reward", continuous_update=False, style=slider_style)
        self.wrong_penalty = widgets.FloatSlider(value=cfg.wrong_penalty, min=-5.0, max=0.0, step=0.05, readout_format=".2f", description="wrong penalty", continuous_update=False, style=slider_style)
        self.timeout_penalty = widgets.FloatSlider(value=cfg.timeout_penalty, min=-5.0, max=0.0, step=0.05, readout_format=".2f", description="timeout penalty", continuous_update=False, style=slider_style)
        self.time_penalty = widgets.FloatLogSlider(value=max(1e-5, cfg.time_penalty), base=10, min=-5, max=0, step=0.1, description="time penalty", continuous_update=False, style=slider_style)
        self.dense_reward_scale = widgets.FloatSlider(value=cfg.dense_reward_scale, min=0.0, max=10.0, step=0.05, readout_format=".3f", description="dense scale", continuous_update=False, style=slider_style)
        self.channel_shaping_scale = widgets.FloatSlider(value=float(getattr(cfg, "channel_shaping_scale", 0.5)), min=0.0, max=5.0, step=0.05, readout_format=".2f", description="shaping scale", continuous_update=False, style=slider_style)
        self.baseline_decay = widgets.FloatSlider(value=cfg.baseline_decay, min=0.0, max=0.999, step=0.005, readout_format=".3f", description="baseline decay", continuous_update=False, style=slider_style)

        self.input_gain = widgets.FloatSlider(value=rcfg.input_gain, min=0.0, max=6.0, step=0.05, description="input gain", continuous_update=False, style=slider_style)
        self.recurrent_scale = widgets.FloatSlider(value=rcfg.recurrent_scale, min=0.0, max=2.5, step=0.01, readout_format=".2f", description="recurrent scale", continuous_update=False, style=slider_style)
        self.decay_rate = widgets.FloatSlider(value=rcfg.decay_rate, min=0.0, max=0.75, step=0.01, readout_format=".2f", description="decay", continuous_update=False, style=slider_style)
        self.spike_threshold = widgets.FloatSlider(value=rcfg.spike_threshold, min=0.05, max=3.0, step=0.05, readout_format=".2f", description="spike threshold", continuous_update=False, style=slider_style)
        self.resting_mp = widgets.FloatSlider(value=rcfg.resting_mp, min=-0.5, max=0.95, step=0.01, readout_format=".2f", description="resting MP", continuous_update=False, style=slider_style)
        self.spike_tau = widgets.FloatSlider(value=rcfg.spike_tau, min=1.0, max=80.0, step=1.0, readout_format=".0f", description="trace tau", continuous_update=False, style=slider_style)
        self.voltage_scale = widgets.FloatLogSlider(value=rcfg.feature_voltage_scale, base=10, min=-2, max=1, step=0.1, description="voltage scale", continuous_update=False, style=slider_style)
        self.pre_steps = widgets.IntSlider(value=rcfg.pre_steps, min=0, max=50, description="pre", continuous_update=False)
        self.on_steps = widgets.IntSlider(value=rcfg.on_steps, min=1, max=120, description="on", continuous_update=False)
        self.off_steps = widgets.IntSlider(value=rcfg.off_steps, min=0, max=80, description="off", continuous_update=False)
        self.readout_start = widgets.IntSlider(value=rcfg.readout_start, min=0, max=180, description="readout start", continuous_update=False, style=slider_style)
        self.readout_stride = widgets.IntSlider(value=rcfg.readout_stride, min=1, max=20, description="readout stride", continuous_update=False, style=slider_style)
        self.reservoir_zmax = widgets.FloatSlider(value=1.0, min=0.05, max=5.0, step=0.05, description="mp zmax", continuous_update=False, style=slider_style)
        self.status = widgets.HTML(value="ready")
        self._sync_all(reset_state=False)
        self._ensure_policy_shape()
        self._refresh_training_figure()
        self._draw_static()

    def _sync_policy_config(self):
        cfg = self.policy.config
        cfg.batch_size = int(self.train_batch_size.value)
        cfg.test_per_digit = int(self.eval_per_digit.value)
        cfg.lr = float(self.lr.value)
        cfg.l2 = float(self.l2.value)
        cfg.decision_threshold = float(np.clip(self.threshold.value, 0.05, 1.0))
        cfg.elimination_threshold = float(np.clip(self.reject_threshold.value, -1.0, -0.05))
        cfg.evidence_gain = float(self.evidence_gain.value)
        cfg.temperature = float(self.temperature.value)
        cfg.exploration_noise = float(self.exploration_noise.value)
        cfg.correct_reward = float(self.correct_reward.value)
        cfg.wrong_penalty = float(self.wrong_penalty.value)
        cfg.timeout_penalty = float(self.timeout_penalty.value)
        cfg.time_penalty = float(self.time_penalty.value)
        cfg.dense_reward_scale = float(self.dense_reward_scale.value)
        cfg.channel_shaping_scale = float(self.channel_shaping_scale.value)
        cfg.baseline_decay = float(self.baseline_decay.value)
        cfg.readout_exclude_input = bool(self.exclude_input_readout.value)
        cfg.readout_neighborhood_hops = int(self.neighborhood_hops.value)
        self.config = replace(cfg)

    def _sync_reservoir_config(self, reset_state: bool = False):
        self.reservoir.apply_runtime_controls(
            input_gain=float(self.input_gain.value),
            recurrent_scale=float(self.recurrent_scale.value),
            decay_rate=float(self.decay_rate.value),
            spike_threshold=float(self.spike_threshold.value),
            resting_mp=float(self.resting_mp.value),
            spike_tau=float(self.spike_tau.value),
            feature_voltage_scale=float(self.voltage_scale.value),
            reset_state=reset_state,
        )
        cfg = self.reservoir.config
        cfg.pre_steps = int(self.pre_steps.value)
        cfg.on_steps = int(self.on_steps.value)
        cfg.off_steps = int(self.off_steps.value)
        cfg.readout_start = int(self.readout_start.value)
        cfg.readout_stride = int(self.readout_stride.value)

    def _sync_all(self, reset_state: bool = False):
        self._sync_policy_config()
        self._sync_reservoir_config(reset_state=reset_state)
        self.feature_mask = self.reservoir.readout_feature_mask(
            exclude_input=bool(self.config.readout_exclude_input),
            neighborhood_hops=int(self.config.readout_neighborhood_hops),
        )

    def _current_feature_mask(self):
        if self.feature_mask is None:
            self.feature_mask = self.reservoir.readout_feature_mask(
                exclude_input=bool(self.config.readout_exclude_input),
                neighborhood_hops=int(self.config.readout_neighborhood_hops),
            )
        return self.feature_mask

    def _mask_feature(self, feature):
        x = np.asarray(feature, dtype=np.float32)
        if x.shape[-1] == self.reservoir.n_features:
            x = x[self._current_feature_mask()]
        return x.astype(np.float32)

    def _mask_sequences(self, sequences):
        X = np.asarray(sequences, dtype=np.float32)
        if X.shape[-1] == self.reservoir.n_features:
            X = X[..., self._current_feature_mask()]
        return X.astype(np.float32)

    def _ensure_policy_shape(self, sequences=None, mask=None):
        n_features = int(np.count_nonzero(self._current_feature_mask()))
        if int(self.policy.n_features) == n_features:
            self._sync_policy_config()
            return False
        old_config = replace(self.policy.config)
        self.policy = EvidenceAccumulatorPolicy(n_features, old_config)
        self.policy.config = replace(self.config)
        if sequences is not None:
            self.policy.set_normalizer(sequences, mask)
        if hasattr(self, "history"):
            self.history = _copy_evidence_history(None)
            globals()["evidence_history"] = self.history
        globals()["evidence_policy"] = self.policy
        return True

    def _clear_handoff_guard(self):
        self.handoff_eval_inputs = None
        self.handoff_eval_labels = None
        self.handoff_eval_sequences = None
        self.handoff_eval_mask = None
        self.handoff_eval_metrics = None
        self.handoff_trial = None
        self.handoff_signature = None
        self.last_guard_status = ""

    def _runtime_signature(self):
        rcfg = self.reservoir.config
        pcfg = self.policy.config
        return (
            float(rcfg.input_gain),
            float(rcfg.recurrent_scale),
            float(rcfg.decay_rate),
            float(rcfg.spike_threshold),
            float(rcfg.resting_mp),
            float(rcfg.spike_tau),
            float(rcfg.feature_voltage_scale),
            int(rcfg.pre_steps),
            int(rcfg.on_steps),
            int(rcfg.off_steps),
            int(rcfg.readout_start),
            int(rcfg.readout_stride),
            bool(pcfg.readout_exclude_input),
            int(pcfg.readout_neighborhood_hops),
            float(pcfg.decision_threshold),
            float(pcfg.elimination_threshold),
            float(pcfg.evidence_gain),
            float(pcfg.correct_reward),
            float(pcfg.wrong_penalty),
            float(pcfg.timeout_penalty),
            float(pcfg.time_penalty),
            float(pcfg.dense_reward_scale),
            int(self.policy.n_features),
        )

    def _snapshot_policy_state(self):
        return {
            "config": replace(self.policy.config),
            "rng_state": copy.deepcopy(self.policy.rng.bit_generator.state),
            "W": self.policy.W.copy(),
            "b": self.policy.b.copy(),
            "mean": self.policy.mean.copy(),
            "std": self.policy.std.copy(),
            "baseline": float(self.policy.baseline),
            "updates": int(self.policy.updates),
            "examples_seen": int(self.policy.examples_seen),
            "last_metrics": dict(self.policy.last_metrics or {}),
        }

    def _restore_policy_state(self, state):
        self.policy.config = replace(state["config"])
        self.policy.rng.bit_generator.state = copy.deepcopy(state["rng_state"])
        self.policy.W = state["W"].copy()
        self.policy.b = state["b"].copy()
        self.policy.mean = state["mean"].copy()
        self.policy.std = state["std"].copy()
        self.policy.baseline = float(state["baseline"])
        self.policy.updates = int(state["updates"])
        self.policy.examples_seen = int(state["examples_seen"])
        self.policy.last_metrics = dict(state["last_metrics"])
        globals()["evidence_policy"] = self.policy

    def _handoff_guard_available(self):
        signature_matches = self.handoff_signature is None or self.handoff_signature == self._runtime_signature()
        return (
            signature_matches
            and self.handoff_eval_sequences is not None
            and self.handoff_eval_labels is not None
            and self.handoff_eval_mask is not None
            and int(self.policy.n_features) == int(np.asarray(self.handoff_eval_sequences).shape[-1])
        )

    def _handoff_guard_metrics(self):
        if not self._handoff_guard_available():
            return None
        return self.policy.evaluate(self.handoff_eval_sequences, self.handoff_eval_labels, self.handoff_eval_mask)

    def _scheduled_eval_metrics(self):
        every_widget = getattr(self, "eval_every_batches", None)
        every = int(every_widget.value) if every_widget is not None else 0
        if every <= 0 or int(self.policy.updates) % every != 0:
            return None
        images, labels = select_balanced(test_images_all, test_labels_all, int(self.eval_per_digit.value), seed=int(self.rng.integers(0, 2**31 - 1)))
        sequences, mask = self._collect_feature_sequences(input_from_images(images), batch_size=128, progress_desc=None)
        return self.policy.evaluate(sequences, labels, mask)

    def _update_policy_batch(self, sequences, labels, mask, *, stochastic=True):
        try:
            return self.policy.update_batch(sequences, labels, mask, stochastic=bool(stochastic))
        except TypeError as exc:
            if "stochastic" not in str(exc):
                raise
            return self.policy.update_batch(sequences, labels, mask)

    def _post_update_metrics(self, sequences, labels, mask, update_metrics, *, rolled_back=False):
        monitor = self.policy.evaluate(sequences, labels, mask)
        metrics = dict(monitor)
        for key in ["policy_loss", "shaping_loss", "grad_norm", "dense_reward"]:
            if key in update_metrics:
                metrics[key] = update_metrics[key]
        metrics["baseline"] = float(self.policy.baseline)
        metrics["updates"] = int(self.policy.updates)
        metrics["examples_seen"] = int(self.policy.examples_seen)
        metrics["rolled_back"] = bool(rolled_back)
        self.last_train_monitor = metrics
        self.policy.last_metrics = dict(metrics)
        return metrics

    def _collect_feature_sequences(self, input_vectors, batch_size=128, progress_desc="evidence sequences"):
        sequences, mask = self.reservoir.collect_feature_sequences(input_vectors, batch_size=batch_size, progress_desc=progress_desc)
        sequences = self._mask_sequences(sequences)
        self._ensure_policy_shape(sequences, mask)
        return sequences, mask

    def _channel_output_from_feature(self, feature, exploration=False):
        x = ((self._mask_feature(feature) - self.policy.mean) / self.policy.std).astype(np.float32)
        out = np.clip(x @ self.policy.W + self.policy.b, -1.0, 1.0).astype(np.float32)
        if exploration and self.policy.config.exploration_noise > 0.0:
            out = np.clip(out + self.policy.rng.normal(0.0, self.policy.config.exploration_noise, size=out.shape), -1.0, 1.0).astype(np.float32)
        return out

    def _history_limit(self):
        widget = getattr(self, "history_points", None)
        return max(1, int(widget.value if widget is not None else 1000))

    def _trim_history(self):
        limit = self._history_limit()
        for key in EVIDENCE_HISTORY_KEYS:
            values = list(self.history.get(key, []))
            if len(values) > limit:
                self.history[key] = values[-limit:]

    def _history_array(self, key):
        x_len = len(self.history.get("updates", []))
        if x_len <= 0:
            return np.asarray([], dtype=np.float32)
        values = list(self.history.get(key, []))[-x_len:]
        if len(values) < x_len:
            values = [np.nan] * (x_len - len(values)) + values
        return np.asarray(values, dtype=np.float32)

    def _record_history(self, metrics, eval_metrics=None):
        self.history["updates"].append(int(metrics.get("updates", self.policy.updates)))
        self.history["train_accuracy"].append(float(metrics.get("accuracy", np.nan)))
        self.history["eval_accuracy"].append(float(eval_metrics.get("accuracy", np.nan)) if eval_metrics else np.nan)
        self.history["reward"].append(float(metrics.get("reward", np.nan)))
        self.history["steps"].append(float(metrics.get("steps", np.nan)))
        self.history["timeout_rate"].append(float(metrics.get("timeout_rate", np.nan)))
        self.history["all_eliminated_rate"].append(float(metrics.get("all_eliminated_rate", np.nan)))
        self.history["policy_loss"].append(float(metrics.get("policy_loss", np.nan)))
        self._trim_history()
        self._refresh_training_figure()

    def _refresh_training_figure(self):
        self._trim_history()
        x = np.asarray(self.history.get("updates", []), dtype=np.float32)
        keys = ["train_accuracy", "eval_accuracy", "reward", "steps", "timeout_rate", "all_eliminated_rate"]
        with self.train_fig.batch_update():
            for idx, key in enumerate(keys):
                self.train_fig.data[idx].x = x
                self.train_fig.data[idx].y = self._history_array(key)

    def _sample_digit(self, digit: int):
        images, labels = (train_images_all, train_labels_all) if self.sample_source.value == "train" else (test_images_all, test_labels_all)
        choices = np.flatnonzero(labels == digit)
        idx = int(self.rng.choice(choices))
        image = images[idx]
        return image, input_from_images(image[None, :, :])[0]

    def _draw_static(self, action=None):
        with self.input_fig.batch_update():
            self.input_fig.data[0].z = self.current_input.reshape(self.reservoir.config.input_shape)
        with self.reservoir_fig.batch_update():
            self.reservoir_fig.data[0].z = self.reservoir.current_frame()
            self.reservoir_fig.data[0].zmax = float(self.reservoir_zmax.value)
        self._draw_evidence(action=action)
        self._draw_accumulation_trace(action=action)

    def _draw_evidence(self, action=None):
        colors = ["#4c78a8" if bool(active) else "#bab0ac" for active in self.current_active]
        if self.current_digit is not None and self.current_active[int(self.current_digit)]:
            colors[int(self.current_digit)] = "#54a24b"
        if action is not None and int(action) >= 0:
            colors[int(action)] = "#f58518"
        with self.evidence_fig.batch_update():
            self.evidence_fig.data[0].y = self.current_evidence
            self.evidence_fig.data[0].marker.color = colors
            self.evidence_fig.data[1].y = [float(self.threshold.value), float(self.threshold.value)]
            self.evidence_fig.data[2].y = [float(self.reject_threshold.value), float(self.reject_threshold.value)]
            self.evidence_fig.layout.yaxis.range = [-1.05, 1.05]
        with self.output_fig.batch_update():
            self.output_fig.data[0].y = self.current_outputs
            self.output_fig.data[0].marker.color = colors

    def _draw_accumulation_trace(self, action=None):
        if self.current_evidence_trace:
            trace = np.asarray(self.current_evidence_trace, dtype=np.float32)
        else:
            trace = np.zeros((1, 10), dtype=np.float32)
        if trace.ndim != 2 or trace.shape[1] != 10:
            trace = np.zeros((1, 10), dtype=np.float32)
        x = np.arange(trace.shape[0], dtype=np.int32)
        x_line = [0, max(1, int(trace.shape[0] - 1))]
        with self.accumulation_fig.batch_update():
            for digit in range(10):
                line = self.accumulation_fig.data[digit]
                line.x = x
                line.y = trace[:, digit]
                line.line.width = 3.0 if (action is not None and int(action) == digit) or (self.current_digit is not None and int(self.current_digit) == digit) else 1.6
                line.opacity = 1.0 if bool(self.current_active[digit]) or trace[-1, digit] > self.policy.config.elimination_threshold else 0.35
            self.accumulation_fig.data[10].x = x_line
            self.accumulation_fig.data[10].y = [float(self.threshold.value), float(self.threshold.value)]
            self.accumulation_fig.data[11].x = x_line
            self.accumulation_fig.data[11].y = [float(self.reject_threshold.value), float(self.reject_threshold.value)]
            self.accumulation_fig.layout.yaxis.range = [-1.05, 1.05]

    def _status_text(self, prefix="ready"):
        stats = self.reservoir.stats()
        last = self.last_rollout or {}
        decision = "n/a" if "action" not in last else f"{last['action']} ({'ok' if last.get('correct') else 'wrong'})"
        eval_text = "n/a" if self.last_eval is None else f"{self.last_eval['accuracy']:.3f}"
        metrics = self.last_train_monitor or self.policy.last_metrics or {}
        guard_text = f" {self.last_guard_status}" if self.last_guard_status else ""
        return (
            f"{prefix} | digit={self.current_digit if self.current_digit is not None else '?'} decision={decision} "
            f"reward={last.get('reward', np.nan):.3f} steps={last.get('steps', 0)} active={int(self.current_active.sum())} "
            f"train_acc={metrics.get('accuracy', np.nan):.3f} eval={eval_text} "
            f"updates={self.policy.updates} seen={self.policy.examples_seen} features={self.policy.n_features} "
            f"recent={stats['recent_spike_fraction']:.3f} ever={stats['ever_spiked_fraction']:.3f} mp_max={stats['mp_max']:.3f}{guard_text}"
        )

    def apply_params(self):
        self.stop_presentation()
        with self._lock:
            self._sync_all(reset_state=False)
            self._ensure_policy_shape()
            self._clear_handoff_guard()
            self.current_evidence.fill(0.0)
            self.current_active.fill(True)
            self.current_outputs.fill(0.0)
            self.current_evidence_trace = [self.current_evidence.copy()]
            self._draw_static()
            self.status.value = self._status_text("params applied")

    def present_digit(self, digit: int):
        self.stop_presentation()
        with self._lock:
            self._sync_all(reset_state=bool(self.reset_before_digit.value))
            self._ensure_policy_shape()
            self.current_digit = int(digit)
            _, self.current_input = self._sample_digit(digit)
            self.current_evidence.fill(0.0)
            self.current_active.fill(True)
            self.current_outputs.fill(0.0)
            self.current_evidence_trace = [self.current_evidence.copy()]
            self.current_sequence = []
            self.sequence_step = 0
            self.readout_count = 0
            self.last_rollout = None
        self._stop.clear()
        self._thread = threading.Thread(target=self._run_presentation, daemon=True, name="mnist-evidence-present")
        self._thread.start()

    def _run_presentation(self):
        cfg = self.reservoir.config
        total_steps = int(cfg.pre_steps + cfg.on_steps + cfg.off_steps)
        action = None
        done = False
        while not self._stop.is_set() and self.sequence_step < total_steps and not done:
            with self._lock:
                for _ in range(int(self.steps_per_frame.value)):
                    if self.sequence_step >= total_steps:
                        break
                    env_value = self.reservoir.envelope(self.sequence_step)
                    self.reservoir.step(self.current_input, sequence_step=self.sequence_step)
                    if self.sequence_step >= cfg.readout_start and (self.sequence_step - cfg.readout_start) % cfg.readout_stride == 0:
                        feature = self.reservoir._snapshot_feature()
                        masked_feature = self._mask_feature(feature)
                        self.current_sequence.append(masked_feature)
                        self.current_outputs = self._channel_output_from_feature(masked_feature, exploration=bool(self.stochastic_present.value))
                        self.current_evidence[self.current_active] = np.clip(
                            self.current_evidence[self.current_active] + self.policy.config.evidence_gain * self.current_outputs[self.current_active],
                            -1.0,
                            1.0,
                        ).astype(np.float32)
                        eliminated = self.current_active & (self.current_evidence <= self.policy.config.elimination_threshold)
                        self.current_active[eliminated] = False
                        self.current_evidence[~self.current_active] = self.policy.config.elimination_threshold
                        self.current_evidence_trace.append(self.current_evidence.copy())
                        self.readout_count += 1
                        active_hits = np.flatnonzero(self.current_active & (self.current_evidence >= self.policy.config.decision_threshold))
                        if len(active_hits):
                            action = int(active_hits[np.argmax(self.current_evidence[active_hits])])
                            done = True
                            break
                        if not self.current_active.any():
                            action = -1
                            done = True
                            break
                    self.sequence_step += 1
                self._draw_static(action=action)
                self.status.value = self._status_text("presenting")
            delay = int(self.delay_ms.value) / 1000.0
            if delay:
                time.sleep(delay)

        with self._lock:
            if not self.current_sequence:
                self.current_sequence.append(self.reservoir._snapshot_feature())
            seq = np.stack(self.current_sequence, axis=0).astype(np.float32)
            mask = np.ones(seq.shape[0], dtype=bool)
            self.last_rollout = self.policy.rollout(seq, int(self.current_digit), mask=mask, train=False, stochastic=False)
            action = int(self.last_rollout["action"])
            self.current_evidence = self.last_rollout["evidence"].astype(np.float32)
            self.current_active = self.last_rollout["active"].astype(bool)
            trace = self.last_rollout.get("evidence_trace", np.empty((0, 10), dtype=np.float32))
            self.current_evidence_trace = [np.zeros(10, dtype=np.float32)] + [np.asarray(row, dtype=np.float32).copy() for row in trace]
            if self.train_on_present.value:
                labels = np.asarray([self.current_digit], dtype=np.int64)
                stochastic = bool(getattr(self, "train_stochastic", None) is None or self.train_stochastic.value)
                update_metrics = self._update_policy_batch(seq[None, :, :], labels, mask[None, :], stochastic=stochastic)
                metrics = self._post_update_metrics(seq[None, :, :], labels, mask[None, :], update_metrics)
                self._record_history(metrics)
            self._draw_static(action=action)
            self.status.value = self._status_text("done")

    def _collect_random_training_batch(self):
        self._sync_all(reset_state=False)
        if bool(self.balanced_batches.value):
            images, labels = sample_balanced_random_labeled(train_images_all, train_labels_all, int(self.train_batch_size.value), self.rng)
        else:
            images, labels = sample_random_labeled(train_images_all, train_labels_all, int(self.train_batch_size.value), self.rng)
        sequences, mask = self._collect_feature_sequences(input_from_images(images), batch_size=min(128, int(self.train_batch_size.value)), progress_desc=None)
        return sequences, labels, mask

    def train_one_batch(self):
        with self._lock:
            sequences, labels, mask = self._collect_random_training_batch()
            guard_enabled = bool(getattr(self, "guard_updates", None) is not None and self.guard_updates.value and self._handoff_guard_available())
            before_state = self._snapshot_policy_state() if guard_enabled else None
            before_guard = self._handoff_guard_metrics() if guard_enabled else None
            stochastic = bool(getattr(self, "train_stochastic", None) is None or self.train_stochastic.value)
            update_metrics = self._update_policy_batch(sequences, labels, mask, stochastic=stochastic)
            rolled_back = False
            eval_metrics = None
            self.last_guard_status = ""
            if guard_enabled and before_guard is not None:
                after_guard = self._handoff_guard_metrics()
                acc_drop = float(before_guard.get("accuracy", np.nan) - after_guard.get("accuracy", np.nan))
                reward_drop = float(before_guard.get("reward", np.nan) - after_guard.get("reward", np.nan))
                if acc_drop > float(self.guard_acc_drop.value) or reward_drop > float(self.guard_reward_drop.value):
                    self._restore_policy_state(before_state)
                    rolled_back = True
                    eval_metrics = before_guard
                    self.last_guard_status = f"guard rollback acc_drop={acc_drop:.3f} reward_drop={reward_drop:.3f}"
                else:
                    eval_metrics = after_guard
                    self.last_guard_status = f"guard ok acc={after_guard['accuracy']:.3f} reward={after_guard['reward']:.3f}"
            if eval_metrics is None:
                eval_metrics = self._scheduled_eval_metrics()
            if eval_metrics is not None:
                self.last_eval = eval_metrics
            metrics = self._post_update_metrics(sequences, labels, mask, update_metrics, rolled_back=rolled_back)
            self._record_history(metrics, eval_metrics=eval_metrics)
            self.last_rollout = None
            suffix = "rolled back" if rolled_back else "trained batch"
            self.status.value = self._status_text(suffix)

    def _run_button_task(self, target, label):
        try:
            target()
        except Exception as exc:
            self._train_stop.set()
            self._stop.set()
            self.status.value = f"{label} failed: {type(exc).__name__}: {exc}"
            traceback.print_exc()

    def train_one_batch_async(self):
        threading.Thread(target=lambda: self._run_button_task(self.train_one_batch, "train batch"), daemon=True, name="mnist-evidence-train-batch").start()

    def train_n_batches(self):
        self._train_stop.clear()
        for _ in range(int(self.train_batches.value)):
            if self._train_stop.is_set():
                break
            self.train_one_batch()

    def train_n_batches_async(self):
        self._start_train_thread(self.train_n_batches)

    def _auto_train_loop(self):
        while not self._train_stop.is_set():
            self.train_one_batch()
            delay = int(self.auto_delay_ms.value) / 1000.0
            if delay:
                time.sleep(delay)

    def _start_train_thread(self, target):
        if self._train_thread is not None and self._train_thread.is_alive():
            return
        self._train_stop.clear()
        self._train_thread = threading.Thread(target=lambda: self._run_button_task(target, "training"), daemon=True, name="mnist-evidence-train")
        self._train_thread.start()

    def start_auto_train(self):
        self._start_train_thread(self._auto_train_loop)

    def evaluate(self):
        with self._lock:
            self._sync_all(reset_state=False)
            images, labels = select_balanced(test_images_all, test_labels_all, int(self.eval_per_digit.value), seed=int(self.rng.integers(0, 2**31 - 1)))
            sequences, mask = self._collect_feature_sequences(input_from_images(images), batch_size=128, progress_desc=None)
            self.last_eval = self.policy.evaluate(sequences, labels, mask)
            metrics = dict(self.policy.last_metrics or {})
            if metrics:
                self._record_history(metrics, eval_metrics=self.last_eval)
            self.status.value = self._status_text("evaluated")

    def evaluate_async(self):
        threading.Thread(target=lambda: self._run_button_task(self.evaluate, "evaluate"), daemon=True, name="mnist-evidence-eval").start()

    def reset_engine(self):
        self.stop_presentation()
        with self._lock:
            self.reservoir.reset()
            self.current_evidence.fill(0.0)
            self.current_active.fill(True)
            self.current_outputs.fill(0.0)
            self.current_evidence_trace = [self.current_evidence.copy()]
            self.current_sequence = []
            self.sequence_step = 0
            self.readout_count = 0
            self._draw_static()
            self.status.value = self._status_text("engine reset")

    def reset_policy(self):
        with self._lock:
            old_mean = self.policy.mean.copy()
            old_std = self.policy.std.copy()
            n_features = int(np.count_nonzero(self._current_feature_mask()))
            self.policy = EvidenceAccumulatorPolicy(n_features, self.config)
            if old_mean.shape[0] == n_features:
                self.policy.mean = old_mean
                self.policy.std = old_std
            self._sync_policy_config()
            self.history = {"updates": [], "train_accuracy": [], "eval_accuracy": [], "reward": [], "steps": [], "timeout_rate": [], "all_eliminated_rate": [], "policy_loss": []}
            self.last_eval = None
            self.last_rollout = None
            self.last_train_monitor = None
            self._clear_handoff_guard()
            self.current_evidence.fill(0.0)
            self.current_active.fill(True)
            self.current_outputs.fill(0.0)
            self.current_evidence_trace = [self.current_evidence.copy()]
            self._refresh_training_figure()
            self._draw_static()
            self.status.value = self._status_text("policy reset")

    def stop_presentation(self):
        self._stop.set()
        if self._thread is not None and self._thread.is_alive():
            self._thread.join(timeout=2)

    def stop_training(self):
        self._train_stop.set()
        if self._train_thread is not None and self._train_thread.is_alive():
            self._train_thread.join(timeout=2)

    def stop_all(self):
        self.stop_presentation()
        self.stop_training()
        self.status.value = "stop requested"

    def display(self):
        controls = widgets.VBox([
            widgets.HBox(self.digit_buttons + [self.train_button, self.train_many_button, self.auto_train_button, self.eval_button, self.stop_button]),
            widgets.HBox([self.reset_engine_button, self.reset_policy_button, self.apply_button, self.sample_source, self.train_on_present, self.balanced_batches, self.reset_before_digit, self.stochastic_present]),
            widgets.HBox([self.exclude_input_readout, self.neighborhood_hops, self.train_stochastic, self.guard_updates]),
            widgets.HBox([self.train_batch_size, self.train_batches, self.eval_per_digit, self.eval_every_batches, self.history_points, self.auto_delay_ms, self.delay_ms, self.steps_per_frame]),
            widgets.HBox([self.guard_acc_drop, self.guard_reward_drop]),
            widgets.HBox([self.lr, self.l2, self.threshold, self.reject_threshold, self.evidence_gain, self.temperature, self.exploration_noise]),
            widgets.HBox([self.correct_reward, self.wrong_penalty, self.timeout_penalty, self.time_penalty, self.dense_reward_scale, self.channel_shaping_scale, self.baseline_decay]),
            widgets.HBox([self.input_gain, self.recurrent_scale, self.decay_rate, self.spike_threshold]),
            widgets.HBox([self.resting_mp, self.spike_tau, self.voltage_scale, self.pre_steps, self.on_steps, self.off_steps]),
            widgets.HBox([self.readout_start, self.readout_stride, self.reservoir_zmax]),
            self.status,
        ])
        display(widgets.VBox([
            controls,
            self.train_fig,
            widgets.HBox([self.input_fig, self.reservoir_fig, self.accumulation_fig]),
            widgets.HBox([self.evidence_fig, self.output_fig]),
        ]))


_previous_evidence_workbench = globals().get("evidence_workbench")
_previous_evidence_history = None
if _previous_evidence_workbench is not None:
    try:
        _previous_evidence_workbench.stop_all()
    except Exception:
        pass
    if hasattr(_previous_evidence_workbench, "policy"):
        evidence_policy = _previous_evidence_workbench.policy
    if hasattr(_previous_evidence_workbench, "history"):
        _previous_evidence_history = _previous_evidence_workbench.history

try:
    evidence_policy
except NameError:
    evidence_policy = _make_default_evidence_policy()

if _previous_evidence_history is not None:
    evidence_history = _previous_evidence_history
elif "evidence_history" in globals() and isinstance(evidence_history, dict):
    evidence_history = evidence_history
elif "rl_history" in globals():
    evidence_history = rl_history
else:
    evidence_history = None

evidence_workbench = EvidenceAccumulatorMNISTWorkbench(reservoir, evidence_policy, getattr(evidence_policy, "config", RL_MNIST_CONFIG), history=evidence_history)
evidence_policy = evidence_workbench.policy
evidence_history = evidence_workbench.history
evidence_workbench.display()


## Evidence Accumulator Optuna Search

This panel searches the same policy and reservoir runtime parameters exposed in the evidence-accumulator workbench, then can apply the best trial back into the live UI. Running the cell only displays controls; the search starts when you click `Run Optuna`.


In [ ]:

try:
    import html
    import threading
    import warnings
    import ipywidgets as widgets
    from IPython.display import display
    try:
        import optuna
        warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)
        optuna.logging.set_verbosity(optuna.logging.WARNING)
    except ImportError as exc:
        optuna = None
        print(f"Optuna is not installed in this environment: {exc}")
except Exception as exc:
    widgets = None
    optuna = None
    print(f"Optuna search UI unavailable: {exc}")


MNIST_EVIDENCE_POLICY_BOUNDS = {
    "lr": (1e-5, 3e-1),
    "l2": (1e-8, 1e-1),
    "decision_threshold": (0.40, 1.25),
    "elimination_threshold": (-1.25, -0.20),
    "evidence_gain": (0.05, 2.0),
    "temperature": (0.02, 1.0),
    "exploration_noise": (0.0, 0.1),
    "correct_reward": (0.50, 3.0),
    "wrong_penalty": (-3.0, -0.20),
    "timeout_penalty": (-2.0, 0.0),
    "time_penalty": (1e-4, 5e-2),
    "dense_reward_scale": (0.0, 10.0),
    "channel_shaping_scale": (0.0, 3.0),
    "baseline_decay": (0.250, 0.995),
    "batch_size": (64,  256),
}

MNIST_EVIDENCE_RESERVOIR_BOUNDS = {
    "input_gain": (0.50, 2.0),
    "recurrent_scale": (0.750, 2.25),
    "decay_rate": (0.001, 0.2),
    "spike_threshold": (0.99, 1.50),
    "resting_mp": (-0.25, 0.1),
    "spike_tau": (2.0, 100.0),
    "feature_voltage_scale": (0.10, 2.0),
    "pre_steps": (0, 5),
    "on_steps": (15, 50),
    "off_steps": (10, 50),
    "readout_start": (20, 60),
    "readout_stride": (2, 6),
}


def _suggest_mnist_evidence_policy_params(trial):
    return {
        "lr": trial.suggest_float("lr", *MNIST_EVIDENCE_POLICY_BOUNDS["lr"], log=True),
        "l2": trial.suggest_float("l2", *MNIST_EVIDENCE_POLICY_BOUNDS["l2"], log=True),
        "decision_threshold": trial.suggest_float("decision_threshold", *MNIST_EVIDENCE_POLICY_BOUNDS["decision_threshold"]),
        "elimination_threshold": trial.suggest_float("elimination_threshold", *MNIST_EVIDENCE_POLICY_BOUNDS["elimination_threshold"]),
        "evidence_gain": trial.suggest_float("evidence_gain", *MNIST_EVIDENCE_POLICY_BOUNDS["evidence_gain"], log=True),
        "temperature": trial.suggest_float("temperature", *MNIST_EVIDENCE_POLICY_BOUNDS["temperature"], log=True),
        "exploration_noise": trial.suggest_float("exploration_noise", *MNIST_EVIDENCE_POLICY_BOUNDS["exploration_noise"]),
        "correct_reward": trial.suggest_float("correct_reward", *MNIST_EVIDENCE_POLICY_BOUNDS["correct_reward"]),
        "wrong_penalty": trial.suggest_float("wrong_penalty", *MNIST_EVIDENCE_POLICY_BOUNDS["wrong_penalty"]),
        "timeout_penalty": trial.suggest_float("timeout_penalty", *MNIST_EVIDENCE_POLICY_BOUNDS["timeout_penalty"]),
        "time_penalty": trial.suggest_float("time_penalty", *MNIST_EVIDENCE_POLICY_BOUNDS["time_penalty"], log=True),
        "dense_reward_scale": trial.suggest_float("dense_reward_scale", *MNIST_EVIDENCE_POLICY_BOUNDS["dense_reward_scale"]),
        "channel_shaping_scale": trial.suggest_float("channel_shaping_scale", *MNIST_EVIDENCE_POLICY_BOUNDS["channel_shaping_scale"]),
        "baseline_decay": trial.suggest_float("baseline_decay", *MNIST_EVIDENCE_POLICY_BOUNDS["baseline_decay"]),
        "batch_size": trial.suggest_categorical("batch_size", list(MNIST_EVIDENCE_POLICY_BOUNDS["batch_size"])),
    }


def _suggest_mnist_evidence_reservoir_params(trial):
    params = {
        "input_gain": trial.suggest_float("input_gain", *MNIST_EVIDENCE_RESERVOIR_BOUNDS["input_gain"], log=True),
        "recurrent_scale": trial.suggest_float("recurrent_scale", *MNIST_EVIDENCE_RESERVOIR_BOUNDS["recurrent_scale"]),
        "decay_rate": trial.suggest_float("decay_rate", *MNIST_EVIDENCE_RESERVOIR_BOUNDS["decay_rate"]),
        "spike_threshold": trial.suggest_float("spike_threshold", *MNIST_EVIDENCE_RESERVOIR_BOUNDS["spike_threshold"]),
        "resting_mp": trial.suggest_float("resting_mp", *MNIST_EVIDENCE_RESERVOIR_BOUNDS["resting_mp"]),
        "spike_tau": trial.suggest_float("spike_tau", *MNIST_EVIDENCE_RESERVOIR_BOUNDS["spike_tau"], log=True),
        "feature_voltage_scale": trial.suggest_float("feature_voltage_scale", *MNIST_EVIDENCE_RESERVOIR_BOUNDS["feature_voltage_scale"], log=True),
        "pre_steps": trial.suggest_int("pre_steps", *MNIST_EVIDENCE_RESERVOIR_BOUNDS["pre_steps"]),
        "on_steps": trial.suggest_int("on_steps", *MNIST_EVIDENCE_RESERVOIR_BOUNDS["on_steps"]),
        "off_steps": trial.suggest_int("off_steps", *MNIST_EVIDENCE_RESERVOIR_BOUNDS["off_steps"]),
        "readout_stride": trial.suggest_int("readout_stride", *MNIST_EVIDENCE_RESERVOIR_BOUNDS["readout_stride"]),
    }
    max_total = params["pre_steps"] + params["on_steps"] + params["off_steps"]
    readout_hi = max(0, min(int(MNIST_EVIDENCE_RESERVOIR_BOUNDS["readout_start"][1]), max_total - 1))
    params["readout_start"] = trial.suggest_int("readout_start", 0, readout_hi)
    return params


def _mnist_evidence_config_from_params(params, base_config: RLMNISTConfig = RL_MNIST_CONFIG, seed: int | None = None, batch_size: int | None = None, epochs: int | None = None):
    cfg_seed = int(seed) if seed is not None else base_config.seed
    return replace(
        base_config,
        seed=cfg_seed,
        lr=float(params.get("lr", base_config.lr)),
        l2=float(params.get("l2", base_config.l2)),
        decision_threshold=float(params.get("decision_threshold", base_config.decision_threshold)),
        elimination_threshold=float(params.get("elimination_threshold", base_config.elimination_threshold)),
        evidence_gain=float(params.get("evidence_gain", base_config.evidence_gain)),
        temperature=float(params.get("temperature", base_config.temperature)),
        exploration_noise=float(params.get("exploration_noise", base_config.exploration_noise)),
        correct_reward=float(params.get("correct_reward", base_config.correct_reward)),
        wrong_penalty=float(params.get("wrong_penalty", base_config.wrong_penalty)),
        timeout_penalty=float(params.get("timeout_penalty", base_config.timeout_penalty)),
        time_penalty=float(params.get("time_penalty", base_config.time_penalty)),
        dense_reward_scale=float(params.get("dense_reward_scale", base_config.dense_reward_scale)),
        channel_shaping_scale=float(params.get("channel_shaping_scale", getattr(base_config, "channel_shaping_scale", 0.5))),
        baseline_decay=float(params.get("baseline_decay", base_config.baseline_decay)),
        readout_exclude_input=bool(params.get("readout_exclude_input", getattr(base_config, "readout_exclude_input", True))),
        readout_neighborhood_hops=int(params.get("readout_neighborhood_hops", getattr(base_config, "readout_neighborhood_hops", 0))),
        batch_size=int(batch_size if batch_size is not None else params.get("batch_size", base_config.batch_size)),
        epochs=int(epochs if epochs is not None else params.get("epochs", base_config.epochs)),
    )


def _cuda_scalar_to_float(value):
    if value is None:
        return None
    if hasattr(value, "get"):
        value = value.get()
    if hasattr(value, "item"):
        value = value.item()
    return float(value)


def _snapshot_mnist_reservoir(reservoir):
    return {
        "config": replace(reservoir.config),
        "U": reservoir.engine.weights.U.copy(),
        "V": reservoir.engine.weights.V.copy(),
        "constant_weight": _cuda_scalar_to_float(reservoir.engine.weights.constant_weight),
        "weights_use_constant_weight": bool(reservoir.engine.weights.use_constant_weight),
        "engine_use_constant_weight": bool(reservoir.engine.use_constant_weight),
        "learning_rate": _cuda_scalar_to_float(reservoir.engine.LEARNING_RATE),
        "weight_summary": reservoir.weight_summary,
    }


def _restore_mnist_reservoir(reservoir, snapshot):
    reservoir.config = replace(snapshot["config"])
    cfg = reservoir.config
    reservoir.engine.weights.U[...] = snapshot["U"]
    reservoir.engine.weights.V[...] = snapshot["V"]
    constant_weight = snapshot["constant_weight"]
    reservoir.engine.weights.constant_weight = None if constant_weight is None else cp.float32(constant_weight)
    reservoir.engine.weights.use_constant_weight = bool(snapshot["weights_use_constant_weight"])
    reservoir.engine.use_constant_weight = bool(snapshot["engine_use_constant_weight"])
    reservoir.engine.LEARNING_RATE = cp.float32(snapshot["learning_rate"])
    reservoir.engine.RESTING_MP = cp.float32(cfg.resting_mp)
    reservoir.engine.DECAY_RATE = cp.float32(cfg.decay_rate)
    reservoir.engine.SPIKE_THRESHOLD = cp.float32(cfg.spike_threshold)
    reservoir.weight_summary = snapshot["weight_summary"]
    reservoir.reset()


def _apply_mnist_reservoir_params(reservoir, params, reset_state: bool = True):
    reservoir.apply_runtime_controls(
        input_gain=float(params.get("input_gain", reservoir.config.input_gain)),
        recurrent_scale=float(params.get("recurrent_scale", reservoir.config.recurrent_scale)),
        decay_rate=float(params.get("decay_rate", reservoir.config.decay_rate)),
        spike_threshold=float(params.get("spike_threshold", reservoir.config.spike_threshold)),
        resting_mp=float(params.get("resting_mp", reservoir.config.resting_mp)),
        spike_tau=float(params.get("spike_tau", reservoir.config.spike_tau)),
        feature_voltage_scale=float(params.get("feature_voltage_scale", reservoir.config.feature_voltage_scale)),
        reset_state=reset_state,
    )
    cfg = reservoir.config
    cfg.pre_steps = int(params.get("pre_steps", cfg.pre_steps))
    cfg.on_steps = int(params.get("on_steps", cfg.on_steps))
    cfg.off_steps = int(params.get("off_steps", cfg.off_steps))
    cfg.readout_start = int(params.get("readout_start", cfg.readout_start))
    cfg.readout_stride = int(params.get("readout_stride", cfg.readout_stride))
    if reset_state:
        reservoir.reset()


def _sparse_odd_pixel_embedding(images, side: int = 28):
    images = np.asarray(images)
    x14 = images[:, ::2, ::2].astype(np.float32) / 255.0
    out = np.zeros((len(images), side, side), dtype=np.float32)
    out[:, 1::2, 1::2] = x14
    return out.reshape(len(images), -1)


def _ridge_direct_metrics(train_images, train_labels, eval_images, eval_labels, ridge: float = 1e-2):
    X_train = _sparse_odd_pixel_embedding(train_images)
    X_eval = _sparse_odd_pixel_embedding(eval_images)
    mean = X_train.mean(axis=0).astype(np.float32)
    std = (X_train.std(axis=0) + 1e-6).astype(np.float32)
    Xn = (X_train - mean) / std
    X_aug = np.column_stack([Xn, np.ones(len(Xn), dtype=np.float32)])
    Y = one_hot(train_labels)
    lhs = X_aug.T @ X_aug
    lhs.flat[:: lhs.shape[0] + 1] += float(ridge)
    lhs[-1, -1] -= float(ridge)
    rhs = X_aug.T @ Y
    weights = np.linalg.solve(lhs.astype(np.float64), rhs.astype(np.float64)).astype(np.float32)
    def predict(X):
        Xn_eval = (X - mean) / std
        X_aug_eval = np.column_stack([Xn_eval, np.ones(len(Xn_eval), dtype=np.float32)])
        return (X_aug_eval @ weights).argmax(axis=1)
    return {
        "accuracy": float(np.mean(predict(X_eval) == np.asarray(eval_labels, dtype=np.int64))),
        "train_accuracy": float(np.mean(predict(X_train) == np.asarray(train_labels, dtype=np.int64))),
        "active_features": int(np.sum(std > 1.1e-6)),
        "total_features": int(X_train.shape[1]),
    }


def _readout_mask_from_params(reservoir, params, workbench=None):
    exclude_input = bool(params.get("readout_exclude_input", True))
    hops = int(params.get("readout_neighborhood_hops", 0))
    if workbench is not None:
        exclude_input = bool(params.get("readout_exclude_input", getattr(workbench.exclude_input_readout, "value", exclude_input)))
        hops = int(params.get("readout_neighborhood_hops", getattr(workbench.neighborhood_hops, "value", hops)))
    mask = reservoir.readout_feature_mask(exclude_input=exclude_input, neighborhood_hops=hops)
    return mask.astype(bool), {"readout_exclude_input": exclude_input, "readout_neighborhood_hops": hops, "readout_features": int(mask.sum())}


def _apply_readout_mask(sequences, feature_mask):
    X = np.asarray(sequences, dtype=np.float32)
    feature_mask = np.asarray(feature_mask, dtype=bool)
    if X.shape[-1] == len(feature_mask):
        return X[..., feature_mask].astype(np.float32)
    return X.astype(np.float32)


def _temporal_shuffle_sequences(sequences, mask, seed: int):
    rng = np.random.default_rng(seed)
    X = np.asarray(sequences, dtype=np.float32).copy()
    M = np.asarray(mask, dtype=bool)
    for i in range(len(X)):
        valid = np.flatnonzero(M[i])
        if len(valid) > 1:
            X[i, valid] = X[i, rng.permutation(valid)]
    return X


def _static_mean_sequences(sequences, mask):
    X = np.asarray(sequences, dtype=np.float32).copy()
    M = np.asarray(mask, dtype=bool)
    for i in range(len(X)):
        valid = M[i]
        if valid.any():
            X[i, valid] = X[i, valid].mean(axis=0, keepdims=True)
    return X


def _first_frame_sequences(sequences, mask):
    X = np.asarray(sequences, dtype=np.float32).copy()
    M = np.asarray(mask, dtype=bool)
    for i in range(len(X)):
        valid = np.flatnonzero(M[i])
        if len(valid):
            X[i, valid] = X[i, valid[0]][None, :]
    return X


def _sequence_dynamics_health(sequences, mask):
    X = np.asarray(sequences, dtype=np.float32)
    M = np.asarray(mask, dtype=bool)
    temporal_stds = []
    temporal_deltas = []
    flat_valid = X[M]
    for seq, seq_mask in zip(X, M):
        valid = seq[seq_mask]
        if len(valid) > 1:
            temporal_stds.append(float(np.mean(np.std(valid, axis=0))))
            temporal_deltas.append(float(np.mean(np.abs(np.diff(valid, axis=0)))))
        elif len(valid) == 1:
            temporal_stds.append(0.0)
            temporal_deltas.append(0.0)
    if len(flat_valid):
        feature_std = np.std(flat_valid, axis=0)
        active_feature_fraction = float(np.mean(feature_std > 1e-4))
        global_std = float(np.mean(feature_std))
    else:
        active_feature_fraction = 0.0
        global_std = 0.0
    return {
        "temporal_std": float(np.mean(temporal_stds)) if temporal_stds else 0.0,
        "temporal_delta": float(np.mean(temporal_deltas)) if temporal_deltas else 0.0,
        "global_feature_std": global_std,
        "active_feature_fraction": active_feature_fraction,
        "readout_steps": float(np.mean(M.sum(axis=1))) if len(M) else 0.0,
    }


def _control_metrics(policy, eval_sequences, eval_labels, eval_mask, direct_metrics, seed: int, no_recurrence_metrics=None):
    shuffled = policy.evaluate(_temporal_shuffle_sequences(eval_sequences, eval_mask, seed=seed), eval_labels, eval_mask)
    static_mean = policy.evaluate(_static_mean_sequences(eval_sequences, eval_mask), eval_labels, eval_mask)
    first_frame = policy.evaluate(_first_frame_sequences(eval_sequences, eval_mask), eval_labels, eval_mask)
    controls = {
        "direct_sparse_pixels": direct_metrics,
        "temporal_shuffle": shuffled,
        "static_mean": static_mean,
        "first_frame": first_frame,
    }
    if no_recurrence_metrics is not None:
        controls["no_recurrence"] = no_recurrence_metrics
    return controls


def _baseline_accuracy_ceiling(control_metrics):
    values = []
    for key, metrics in (control_metrics or {}).items():
        if isinstance(metrics, dict) and "accuracy" in metrics:
            values.append(float(metrics["accuracy"]))
    return max(values) if values else 0.0


def _train_mnist_evidence_candidate(policy, train_sequences, train_labels, train_mask, eval_sequences, eval_labels, eval_mask, trial=None, stop_event=None):
    rng = np.random.default_rng(policy.config.seed)
    history = {"epoch": [], "update": [], "train_accuracy": [], "eval_accuracy": [], "reward": [], "steps": [], "timeout_rate": [], "all_eliminated_rate": [], "policy_loss": []}
    labels = np.asarray(train_labels, dtype=np.int64)
    for epoch in range(int(policy.config.epochs)):
        if stop_event is not None and stop_event.is_set():
            raise optuna.TrialPruned()
        order = rng.permutation(len(labels))
        last_metrics = None
        for start in range(0, len(order), int(policy.config.batch_size)):
            if stop_event is not None and stop_event.is_set():
                raise optuna.TrialPruned()
            idx = order[start:start + int(policy.config.batch_size)]
            last_metrics = policy.update_batch(train_sequences[idx], labels[idx], train_mask[idx])
        eval_metrics = policy.evaluate(eval_sequences, eval_labels, eval_mask)
        score = _score_mnist_evidence_eval(eval_metrics, max_steps=train_sequences.shape[1])
        if trial is not None:
            trial.report(score, step=epoch + 1)
            if trial.should_prune():
                raise optuna.TrialPruned()
        history["epoch"].append(epoch + 1)
        history["update"].append(policy.updates)
        history["train_accuracy"].append(last_metrics["accuracy"] if last_metrics else np.nan)
        history["eval_accuracy"].append(eval_metrics["accuracy"])
        history["reward"].append(last_metrics["reward"] if last_metrics else np.nan)
        history["steps"].append(eval_metrics["steps"])
        history["timeout_rate"].append(eval_metrics["timeout_rate"])
        history["all_eliminated_rate"].append(eval_metrics["all_eliminated_rate"])
        history["policy_loss"].append(last_metrics["policy_loss"] if last_metrics else np.nan)
    return history, eval_metrics


def _score_mnist_evidence_eval(metrics, max_steps: int, control_metrics=None, health=None):
    max_steps = max(1, int(max_steps))
    accuracy = float(metrics.get("accuracy", 0.0))
    step_cost = float(metrics.get("steps", max_steps)) / max_steps
    baseline_ceiling = _baseline_accuracy_ceiling(control_metrics)
    dynamic_advantage = accuracy - baseline_ceiling
    health = health or {}
    temporal_std = float(health.get("temporal_std", 0.0))
    temporal_delta = float(health.get("temporal_delta", 0.0))
    active_fraction = float(health.get("active_feature_fraction", 0.0))
    low_dynamics_penalty = max(0.0, 1e-3 - temporal_std) * 50.0 + max(0.0, 1e-4 - temporal_delta) * 100.0
    inactive_penalty = max(0.0, 0.05 - active_fraction)
    return float(
        accuracy
        + 0.50 * dynamic_advantage
        + (1.50 * dynamic_advantage if dynamic_advantage < 0.0 else 0.0)
        - 0.10 * step_cost
        - 0.20 * metrics.get("timeout_rate", 0.0)
        - 0.20 * metrics.get("all_eliminated_rate", 0.0)
        - low_dynamics_penalty
        - inactive_penalty
        + 0.01 * metrics.get("reward", 0.0)
    )


def evaluate_mnist_evidence_candidate(workbench, params, train_per_digit: int, eval_per_digit: int, epochs: int, batch_size: int, seed: int, optimize_reservoir: bool = True, run_no_recurrence_ablation: bool = False, trial=None, stop_event=None):
    reservoir = workbench.reservoir
    params = dict(params)
    params.setdefault("readout_exclude_input", True)
    params.setdefault("readout_neighborhood_hops", int(getattr(getattr(workbench, "neighborhood_hops", None), "value", 0)))
    if optimize_reservoir:
        _apply_mnist_reservoir_params(reservoir, params, reset_state=True)
    images, labels = select_balanced(train_images_all, train_labels_all, int(train_per_digit), seed=int(seed))
    eval_images, eval_labels = select_balanced(test_images_all, test_labels_all, int(eval_per_digit), seed=int(seed) + 1)
    direct_metrics = _ridge_direct_metrics(images, labels, eval_images, eval_labels, ridge=1e-2)
    feature_mask, readout_meta = _readout_mask_from_params(reservoir, params, workbench=workbench)
    params.update(readout_meta)
    if int(readout_meta["readout_features"]) < 10:
        if optuna is not None:
            raise optuna.TrialPruned()
        raise ValueError("Readout mask left fewer than 10 features.")
    trial_batch_size = int(params.get("batch_size", batch_size))
    feature_batch_size = int(getattr(reservoir.config, "feature_batch_size", 128))
    train_sequences_full, train_mask = reservoir.collect_feature_sequences(input_from_images(images), batch_size=feature_batch_size, progress_desc=None)
    eval_sequences_full, eval_mask = reservoir.collect_feature_sequences(input_from_images(eval_images), batch_size=feature_batch_size, progress_desc=None)
    train_sequences = _apply_readout_mask(train_sequences_full, feature_mask)
    eval_sequences = _apply_readout_mask(eval_sequences_full, feature_mask)
    health = _sequence_dynamics_health(train_sequences, train_mask)
    cfg = _mnist_evidence_config_from_params(params, base_config=workbench.policy.config, seed=int(seed), batch_size=trial_batch_size, epochs=int(epochs))
    policy = EvidenceAccumulatorPolicy(train_sequences.shape[-1], cfg)
    policy.set_normalizer(train_sequences, train_mask)
    history, eval_metrics = _train_mnist_evidence_candidate(policy, train_sequences, labels, train_mask, eval_sequences, eval_labels, eval_mask, trial=trial, stop_event=stop_event)

    no_recurrence_metrics = None
    if run_no_recurrence_ablation:
        no_recurrence_snapshot = _snapshot_mnist_reservoir(reservoir)
        try:
            no_recur_params = dict(params)
            no_recur_params["recurrent_scale"] = 0.0
            _apply_mnist_reservoir_params(reservoir, no_recur_params, reset_state=True)
            no_train_full, no_train_mask = reservoir.collect_feature_sequences(input_from_images(images), batch_size=feature_batch_size, progress_desc=None)
            no_eval_full, no_eval_mask = reservoir.collect_feature_sequences(input_from_images(eval_images), batch_size=feature_batch_size, progress_desc=None)
            no_train = _apply_readout_mask(no_train_full, feature_mask)
            no_eval = _apply_readout_mask(no_eval_full, feature_mask)
            no_policy = EvidenceAccumulatorPolicy(no_train.shape[-1], replace(cfg, seed=int(seed) + 17))
            no_policy.set_normalizer(no_train, no_train_mask)
            _train_mnist_evidence_candidate(no_policy, no_train, labels, no_train_mask, no_eval, eval_labels, no_eval_mask, trial=None, stop_event=stop_event)
            no_recurrence_metrics = no_policy.evaluate(no_eval, eval_labels, no_eval_mask)
        finally:
            _restore_mnist_reservoir(reservoir, no_recurrence_snapshot)

    controls = _control_metrics(policy, eval_sequences, eval_labels, eval_mask, direct_metrics, seed=int(seed) + 2, no_recurrence_metrics=no_recurrence_metrics)
    baseline_accuracy = _baseline_accuracy_ceiling(controls)
    dynamic_advantage = float(eval_metrics.get("accuracy", 0.0) - baseline_accuracy)
    final_score = _score_mnist_evidence_eval(eval_metrics, max_steps=train_sequences.shape[1], control_metrics=controls, health=health)
    return {
        "score": final_score,
        "eval_metrics": eval_metrics,
        "control_metrics": controls,
        "health": health,
        "baseline_accuracy": baseline_accuracy,
        "dynamic_advantage": dynamic_advantage,
        "readout_meta": readout_meta,
        "history": history,
        "params": dict(params),
        "eval_inputs": input_from_images(eval_images).astype(np.float32).copy(),
        "eval_labels": np.asarray(eval_labels, dtype=np.int64).copy(),
        "policy_state": {
            "config": replace(policy.config),
            "rng_state": policy.rng.bit_generator.state,
            "W": policy.W.copy(),
            "b": policy.b.copy(),
            "mean": policy.mean.copy(),
            "std": policy.std.copy(),
            "baseline": float(policy.baseline),
            "updates": int(policy.updates),
            "examples_seen": int(policy.examples_seen),
            "last_metrics": dict(policy.last_metrics),
        },
    }


def run_mnist_evidence_optuna_search(workbench, n_trials: int = 20, train_per_digit: int = 10, eval_per_digit: int = 5, epochs: int = 2, batch_size: int = 32, seed: int = 123, optimize_reservoir: bool = True, run_no_recurrence_ablation: bool = False, readout_exclude_input: bool = True, readout_neighborhood_hops: int = 0, study_name: str | None = None, storage: str | None = None, load_if_exists: bool = True, stop_event=None, status_callback=None):
    if optuna is None:
        raise RuntimeError("Install optuna to run the parameter search.")
    workbench.stop_all()
    reservoir_snapshot = _snapshot_mnist_reservoir(workbench.reservoir)
    trial_results = {}
    startup = max(4, min(10, int(n_trials) // 3 if int(n_trials) >= 9 else int(n_trials)))
    sampler = optuna.samplers.TPESampler(seed=int(seed), n_startup_trials=startup, multivariate=True, group=True)
    pruner = optuna.pruners.NopPruner()
    study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner, study_name=study_name, storage=storage, load_if_exists=load_if_exists)

    def objective(trial):
        if stop_event is not None and stop_event.is_set():
            raise optuna.TrialPruned()
        _restore_mnist_reservoir(workbench.reservoir, reservoir_snapshot)
        params = _suggest_mnist_evidence_policy_params(trial)
        if optimize_reservoir:
            params.update(_suggest_mnist_evidence_reservoir_params(trial))
        params["readout_exclude_input"] = bool(readout_exclude_input)
        params["readout_neighborhood_hops"] = int(readout_neighborhood_hops)
        trial_seed = int(seed) * 10_000 + trial.number
        result = evaluate_mnist_evidence_candidate(
            workbench,
            params,
            train_per_digit=int(train_per_digit),
            eval_per_digit=int(eval_per_digit),
            epochs=int(epochs),
            batch_size=int(batch_size),
            seed=trial_seed,
            optimize_reservoir=bool(optimize_reservoir),
            run_no_recurrence_ablation=bool(run_no_recurrence_ablation),
            trial=trial,
            stop_event=stop_event,
        )
        trial_results[trial.number] = result
        metrics = result["eval_metrics"]
        trial.set_user_attr("eval_accuracy", float(metrics["accuracy"]))
        trial.set_user_attr("eval_reward", float(metrics["reward"]))
        trial.set_user_attr("eval_steps", float(metrics["steps"]))
        trial.set_user_attr("timeout_rate", float(metrics["timeout_rate"]))
        trial.set_user_attr("all_eliminated_rate", float(metrics["all_eliminated_rate"]))
        trial.set_user_attr("baseline_accuracy", float(result.get("baseline_accuracy", np.nan)))
        trial.set_user_attr("dynamic_advantage", float(result.get("dynamic_advantage", np.nan)))
        trial.set_user_attr("direct_accuracy", float(result.get("control_metrics", {}).get("direct_sparse_pixels", {}).get("accuracy", np.nan)))
        trial.set_user_attr("temporal_shuffle_accuracy", float(result.get("control_metrics", {}).get("temporal_shuffle", {}).get("accuracy", np.nan)))
        trial.set_user_attr("static_mean_accuracy", float(result.get("control_metrics", {}).get("static_mean", {}).get("accuracy", np.nan)))
        trial.set_user_attr("first_frame_accuracy", float(result.get("control_metrics", {}).get("first_frame", {}).get("accuracy", np.nan)))
        trial.set_user_attr("no_recurrence_accuracy", float(result.get("control_metrics", {}).get("no_recurrence", {}).get("accuracy", np.nan)))
        trial.set_user_attr("temporal_std", float(result.get("health", {}).get("temporal_std", np.nan)))
        trial.set_user_attr("temporal_delta", float(result.get("health", {}).get("temporal_delta", np.nan)))
        trial.set_user_attr("active_feature_fraction", float(result.get("health", {}).get("active_feature_fraction", np.nan)))
        trial.set_user_attr("readout_features", int(result.get("readout_meta", {}).get("readout_features", 0)))
        trial.set_user_attr("seed", trial_seed)
        return result["score"]

    def update_progress(study, trial):
        if stop_event is not None and stop_event.is_set():
            study.stop()
        if status_callback is not None:
            finished = len([t for t in study.trials if t.state.is_finished()])
            complete = len(_complete_mnist_trials(study))
            pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
            best = "n/a"
            try:
                best = f"{study.best_value:.4f}"
            except ValueError:
                pass
            status_callback(f"Optuna trial {finished}/{int(n_trials)} | complete={complete} pruned={pruned} | last={trial.state.name} | best={best}")

    try:
        study.optimize(objective, n_trials=int(n_trials), callbacks=[update_progress], show_progress_bar=False, gc_after_trial=True)
    finally:
        _restore_mnist_reservoir(workbench.reservoir, reservoir_snapshot)
    return study, mnist_evidence_optuna_results(study, trial_results)


def _is_finite_number(value):
    try:
        return bool(np.isfinite(float(value)))
    except (TypeError, ValueError):
        return False


def _complete_mnist_trials(study):
    complete_state = optuna.trial.TrialState.COMPLETE if optuna is not None else None
    return [trial for trial in study.trials if complete_state is None or trial.state == complete_state]


def mnist_evidence_optuna_results(study, trial_results=None):
    trial_results = trial_results or {}
    rows = []
    for trial in _complete_mnist_trials(study):
        stored = trial_results.get(trial.number, {})
        if trial.value is None or not _is_finite_number(trial.value):
            continue
        if not stored and not _is_finite_number(trial.user_attrs.get("eval_accuracy", np.nan)):
            continue
        metrics = {
            "accuracy": stored.get("eval_metrics", {}).get("accuracy", trial.user_attrs.get("eval_accuracy", np.nan)),
            "reward": stored.get("eval_metrics", {}).get("reward", trial.user_attrs.get("eval_reward", np.nan)),
            "steps": stored.get("eval_metrics", {}).get("steps", trial.user_attrs.get("eval_steps", np.nan)),
            "timeout_rate": stored.get("eval_metrics", {}).get("timeout_rate", trial.user_attrs.get("timeout_rate", np.nan)),
            "all_eliminated_rate": stored.get("eval_metrics", {}).get("all_eliminated_rate", trial.user_attrs.get("all_eliminated_rate", np.nan)),
        }
        if not _is_finite_number(metrics["accuracy"]):
            continue
        control_metrics = stored.get("control_metrics") or {
            "direct_sparse_pixels": {"accuracy": trial.user_attrs.get("direct_accuracy", np.nan)},
            "temporal_shuffle": {"accuracy": trial.user_attrs.get("temporal_shuffle_accuracy", np.nan)},
            "static_mean": {"accuracy": trial.user_attrs.get("static_mean_accuracy", np.nan)},
            "first_frame": {"accuracy": trial.user_attrs.get("first_frame_accuracy", np.nan)},
            "no_recurrence": {"accuracy": trial.user_attrs.get("no_recurrence_accuracy", np.nan)},
        }
        health = stored.get("health") or {
            "temporal_std": trial.user_attrs.get("temporal_std", np.nan),
            "temporal_delta": trial.user_attrs.get("temporal_delta", np.nan),
            "active_feature_fraction": trial.user_attrs.get("active_feature_fraction", np.nan),
        }
        rows.append({
            "trial": int(trial.number),
            "score": float(trial.value),
            "params": dict(stored.get("params") or trial.params),
            "eval_metrics": metrics,
            "control_metrics": control_metrics,
            "health": health,
            "baseline_accuracy": float(stored.get("baseline_accuracy", trial.user_attrs.get("baseline_accuracy", np.nan))),
            "dynamic_advantage": float(stored.get("dynamic_advantage", trial.user_attrs.get("dynamic_advantage", np.nan))),
            "readout_meta": stored.get("readout_meta") or {"readout_features": trial.user_attrs.get("readout_features", np.nan)},
            "history": stored.get("history"),
            "eval_inputs": stored.get("eval_inputs"),
            "eval_labels": stored.get("eval_labels"),
            "policy_state": stored.get("policy_state"),
        })
    return sorted(rows, key=lambda row: row["score"], reverse=True)


def _widget_bounds(widget):
    if isinstance(widget, widgets.FloatLogSlider):
        base = float(widget.base)
        return base ** float(widget.min), base ** float(widget.max)
    if hasattr(widget, "min") and hasattr(widget, "max"):
        return widget.min, widget.max
    return None, None


def _coerce_widget_value(widget, value):
    lo, hi = _widget_bounds(widget)
    if lo is not None:
        value = max(lo, value)
    if hi is not None:
        value = min(hi, value)
    if isinstance(widget.value, (int, np.integer)) and not isinstance(widget.value, bool):
        return int(round(value))
    return float(value)


def _set_widget_value(widget, value):
    widget.value = _coerce_widget_value(widget, value)


def _widget_numeric_value(widget):
    return float(widget.value)


def _validate_mnist_optuna_result_applied(result, workbench, widget_map, *, atol: float = 1e-7):
    mismatches = []
    for key, widget in widget_map.items():
        if key not in result["params"]:
            continue
        expected = _coerce_widget_value(widget, float(result["params"][key]))
        actual = _widget_numeric_value(widget)
        tolerance = max(atol, 1e-5 * max(1.0, abs(expected)))
        if abs(actual - float(expected)) > tolerance:
            mismatches.append((key, float(result["params"][key]), float(expected), actual))
    if mismatches:
        detail = "; ".join(f"{key}: optuna={raw} expected_widget={expected} actual={actual}" for key, raw, expected, actual in mismatches)
        raise RuntimeError("Optuna result did not apply cleanly to the workbench sliders: " + detail)


def _verify_mnist_optuna_handoff(result, workbench):
    eval_inputs = result.get("eval_inputs")
    eval_labels = result.get("eval_labels")
    if eval_inputs is None or eval_labels is None:
        return None
    if hasattr(workbench, "_collect_feature_sequences"):
        sequences, mask = workbench._collect_feature_sequences(
            np.asarray(eval_inputs, dtype=np.float32),
            batch_size=128,
            progress_desc=None,
        )
    else:
        sequences, mask = workbench.reservoir.collect_feature_sequences(
            np.asarray(eval_inputs, dtype=np.float32),
            batch_size=128,
            progress_desc=None,
        )
    eval_labels_array = np.asarray(eval_labels, dtype=np.int64)
    metrics = workbench.policy.evaluate(sequences, eval_labels_array, mask)
    if hasattr(workbench, "handoff_eval_sequences"):
        workbench.handoff_eval_inputs = np.asarray(eval_inputs, dtype=np.float32).copy()
        workbench.handoff_eval_labels = eval_labels_array.copy()
        workbench.handoff_eval_sequences = np.asarray(sequences, dtype=np.float32).copy()
        workbench.handoff_eval_mask = np.asarray(mask, dtype=bool).copy()
        workbench.handoff_eval_metrics = dict(metrics)
        workbench.handoff_trial = result.get("trial")
        if hasattr(workbench, "_runtime_signature"):
            workbench.handoff_signature = workbench._runtime_signature()
        workbench.last_guard_status = f"guard set trial={result.get('trial', '?')} acc={metrics['accuracy']:.3f}"
    expected = result.get("eval_metrics", {}) or {}
    if "accuracy" in expected:
        tolerance = max(1.0 / max(1, len(eval_labels)), 1e-6)
        if abs(float(metrics["accuracy"]) - float(expected["accuracy"])) > tolerance:
            raise RuntimeError(
                "Applied Optuna policy does not reproduce the trial evaluation: "
                f"expected accuracy {float(expected['accuracy']):.4f}, got {float(metrics['accuracy']):.4f}."
            )
    workbench.last_eval = metrics
    return metrics


def apply_mnist_evidence_optuna_result(result, workbench=None, install_policy: bool = True):
    if workbench is None:
        workbench = evidence_workbench
    params = result["params"]
    widget_map = {
        "lr": workbench.lr,
        "l2": workbench.l2,
        "decision_threshold": workbench.threshold,
        "elimination_threshold": workbench.reject_threshold,
        "evidence_gain": workbench.evidence_gain,
        "temperature": workbench.temperature,
        "exploration_noise": workbench.exploration_noise,
        "correct_reward": workbench.correct_reward,
        "wrong_penalty": workbench.wrong_penalty,
        "timeout_penalty": workbench.timeout_penalty,
        "time_penalty": workbench.time_penalty,
        "dense_reward_scale": workbench.dense_reward_scale,
        "channel_shaping_scale": workbench.channel_shaping_scale,
        "baseline_decay": workbench.baseline_decay,
        "batch_size": workbench.train_batch_size,
        "input_gain": workbench.input_gain,
        "recurrent_scale": workbench.recurrent_scale,
        "decay_rate": workbench.decay_rate,
        "spike_threshold": workbench.spike_threshold,
        "resting_mp": workbench.resting_mp,
        "spike_tau": workbench.spike_tau,
        "feature_voltage_scale": workbench.voltage_scale,
        "pre_steps": workbench.pre_steps,
        "on_steps": workbench.on_steps,
        "off_steps": workbench.off_steps,
        "readout_start": workbench.readout_start,
        "readout_stride": workbench.readout_stride,
        "readout_neighborhood_hops": workbench.neighborhood_hops,
    }
    workbench.stop_all()
    if hasattr(workbench, "train_on_present"):
        workbench.train_on_present.value = False
    if hasattr(workbench, "exclude_input_readout"):
        workbench.exclude_input_readout.value = bool(params.get("readout_exclude_input", True))
    for key, widget in widget_map.items():
        if key in params:
            _set_widget_value(widget, float(params[key]))
    _validate_mnist_optuna_result_applied(result, workbench, widget_map)
    workbench.apply_params()
    if install_policy and result.get("policy_state") is not None:
        state = result["policy_state"]
        state_config = state.get("config")
        if state_config is not None:
            cfg = replace(state_config)
        else:
            cfg = _mnist_evidence_config_from_params(params, base_config=workbench.policy.config, seed=workbench.policy.config.seed, batch_size=workbench.policy.config.batch_size, epochs=workbench.policy.config.epochs)
        state_features = int(state["W"].shape[0])
        active_features = int(np.count_nonzero(workbench._current_feature_mask())) if hasattr(workbench, "_current_feature_mask") else state_features
        if state_features != active_features:
            raise RuntimeError(
                "Optuna policy feature count does not match the active workbench readout mask: "
                f"policy has {state_features}, mask exposes {active_features}. Re-run Optuna with the current notebook cells."
            )
        policy = EvidenceAccumulatorPolicy(state_features, cfg)
        policy.W = state["W"].copy()
        policy.b = state["b"].copy()
        policy.mean = state["mean"].copy()
        policy.std = state["std"].copy()
        if state.get("rng_state") is not None:
            policy.rng.bit_generator.state = state["rng_state"]
        policy.baseline = float(state["baseline"])
        policy.updates = int(state["updates"])
        policy.examples_seen = int(state["examples_seen"])
        policy.last_metrics = dict(state.get("last_metrics", {}))
        workbench.policy = policy
        globals()["evidence_policy"] = workbench.policy
        if result.get("history") is not None:
            if "_history_with_policy_tail" in globals():
                workbench.history = _history_with_policy_tail(result["history"], workbench.policy)
            elif "_copy_evidence_history" in globals():
                workbench.history = _copy_evidence_history(result["history"])
            else:
                workbench.history = {key: list(result["history"].get(key, [])) for key in ["updates", "train_accuracy", "eval_accuracy", "reward", "steps", "timeout_rate", "all_eliminated_rate", "policy_loss"]}
            globals()["evidence_history"] = workbench.history
        workbench._sync_policy_config()
        workbench.current_evidence.fill(0.0)
        workbench.current_active.fill(True)
        workbench.current_outputs.fill(0.0)
        workbench.current_evidence_trace = [workbench.current_evidence.copy()]
        workbench.last_rollout = None
        handoff_metrics = _verify_mnist_optuna_handoff(result, workbench)
        workbench._refresh_training_figure()
        workbench._draw_static()
        if handoff_metrics is None:
            workbench.status.value = workbench._status_text("optuna best applied")
        else:
            workbench.status.value = workbench._status_text(f"optuna best applied | verified_acc={handoff_metrics['accuracy']:.3f}")
    _validate_mnist_optuna_result_applied(result, workbench, widget_map)
    return workbench.policy


def _mnist_optuna_results_table(results, limit: int = 10):
    if not results:
        return "<em>No completed trials yet.</em>"
    headers = ["trial", "score", "acc", "dyn+", "base", "direct", "shuffle", "static", "first", "no-rec", "tstd", "active", "features", "hops", "dense", "shape", "gain", "lr", "batch"]
    rows = ["<table><thead><tr>" + "".join(f"<th>{h}</th>" for h in headers) + "</tr></thead><tbody>"]
    shown = 0
    for row in results:
        if shown >= int(limit):
            break
        metrics = row["eval_metrics"]
        if not _is_finite_number(metrics.get("accuracy", np.nan)):
            continue
        params = row["params"]
        controls = row.get("control_metrics", {}) or {}
        health = row.get("health", {}) or {}
        direct = controls.get("direct_sparse_pixels", {}) or {}
        shuffle = controls.get("temporal_shuffle", {}) or {}
        static = controls.get("static_mean", {}) or {}
        first = controls.get("first_frame", {}) or {}
        no_recur = controls.get("no_recurrence", {}) or {}
        values = [
            row["trial"],
            f"{row['score']:.4f}",
            f"{metrics.get('accuracy', np.nan):.3f}",
            f"{row.get('dynamic_advantage', np.nan):+.3f}",
            f"{row.get('baseline_accuracy', np.nan):.3f}",
            f"{direct.get('accuracy', np.nan):.3f}",
            f"{shuffle.get('accuracy', np.nan):.3f}",
            f"{static.get('accuracy', np.nan):.3f}",
            f"{first.get('accuracy', np.nan):.3f}",
            f"{no_recur.get('accuracy', np.nan):.3f}",
            f"{health.get('temporal_std', np.nan):.2e}",
            f"{health.get('active_feature_fraction', np.nan):.2f}",
            row.get("readout_meta", {}).get("readout_features", params.get("readout_features", "")),
            params.get("readout_neighborhood_hops", ""),
            f"{params.get('dense_reward_scale', np.nan):.3g}",
            f"{params.get('channel_shaping_scale', np.nan):.3g}",
            f"{params.get('evidence_gain', np.nan):.3g}",
            f"{params.get('lr', np.nan):.3g}",
            params.get("batch_size", ""),
        ]
        rows.append("<tr>" + "".join(f"<td>{html.escape(str(v))}</td>" for v in values) + "</tr>")
        shown += 1
    if shown == 0:
        return "<em>No completed trials with final metrics yet.</em>"
    rows.append("</tbody></table>")
    return "".join(rows)


class EvidenceOptunaSearchPanel:
    def __init__(self, workbench):
        if widgets is None:
            raise RuntimeError("ipywidgets is required for the Optuna panel")
        self.workbench = workbench
        self.study = None
        self.results = []
        self._thread = None
        self._stop = threading.Event()
        self._active_settings = None
        style = {"description_width": "initial"}
        self.trials = widgets.IntSlider(value=16, min=1, max=200, step=1, description="trials", continuous_update=False, style=style)
        self.train_per_digit = widgets.IntSlider(value=10, min=1, max=300, step=1, description="train/digit", continuous_update=False, style=style)
        self.eval_per_digit = widgets.IntSlider(value=5, min=1, max=200, step=1, description="eval/digit", continuous_update=False, style=style)
        self.epochs = widgets.IntSlider(value=2, min=1, max=30, step=1, description="epochs/trial", continuous_update=False, style=style)
        self.batch_size = widgets.IntSlider(value=32, min=1, max=512, step=1, description="batch", continuous_update=False, style=style)
        self.seed = widgets.IntText(value=123, description="seed", style=style)
        self.optimize_reservoir = widgets.Checkbox(value=True, description="search reservoir controls", style=style)
        self.no_recurrence_ablation = widgets.Checkbox(value=False, description="no-recurrence ablation", style=style)
        self.mask_input_readout = widgets.Checkbox(value=bool(getattr(workbench.exclude_input_readout, "value", True)), description="mask input readout", style=style)
        self.mask_hops = widgets.IntSlider(value=int(getattr(workbench.neighborhood_hops, "value", 0)), min=0, max=3, step=1, description="mask hops", continuous_update=False, style=style)
        self.install_policy = widgets.Checkbox(value=True, description="install trained policy", style=style)
        self.run_button = widgets.Button(description="Run Optuna", icon="search", button_style="success")
        self.stop_button = widgets.Button(description="Stop", icon="stop", button_style="danger")
        self.apply_button = widgets.Button(description="Apply Best", icon="check", button_style="primary")
        self.status = widgets.HTML(value="ready")
        self.table = widgets.HTML(value="<em>No completed trials yet.</em>")
        self.run_button.on_click(lambda _: self.run_async())
        self.stop_button.on_click(lambda _: self.stop())
        self.apply_button.on_click(lambda _: self.apply_best())
        self._search_controls = [self.trials, self.train_per_digit, self.eval_per_digit, self.epochs, self.batch_size, self.seed, self.optimize_reservoir, self.no_recurrence_ablation, self.mask_input_readout, self.mask_hops]
        self._set_running(False)

    def _set_status(self, text):
        self.status.value = html.escape(str(text))

    def _search_settings(self):
        return {
            "n_trials": int(self.trials.value),
            "train_per_digit": int(self.train_per_digit.value),
            "eval_per_digit": int(self.eval_per_digit.value),
            "epochs": int(self.epochs.value),
            "batch_size": int(self.batch_size.value),
            "seed": int(self.seed.value),
            "optimize_reservoir": bool(self.optimize_reservoir.value),
            "run_no_recurrence_ablation": bool(self.no_recurrence_ablation.value),
            "readout_exclude_input": bool(self.mask_input_readout.value),
            "readout_neighborhood_hops": int(self.mask_hops.value),
        }

    def _settings_summary(self, settings):
        return (
            f"trials={settings['n_trials']} train/digit={settings['train_per_digit']} "
            f"eval/digit={settings['eval_per_digit']} epochs={settings['epochs']} batch={settings['batch_size']} "
            f"mask_input={settings['readout_exclude_input']} hops={settings['readout_neighborhood_hops']} "
            f"no_recur={settings['run_no_recurrence_ablation']}"
        )

    def _set_running(self, running: bool):
        running = bool(running)
        for control in self._search_controls:
            control.disabled = running
        self.run_button.disabled = running
        self.apply_button.disabled = running
        self.stop_button.disabled = not running

    def run_async(self):
        if optuna is None:
            self._set_status("Optuna is not installed in this environment.")
            return
        if self._thread is not None and self._thread.is_alive():
            suffix = "" if self._active_settings is None else f" | active {self._settings_summary(self._active_settings)}"
            self._set_status("search already running" + suffix)
            return
        settings = self._search_settings()
        self._active_settings = settings
        self._stop.clear()
        self._set_running(True)
        self._set_status("starting Optuna search | " + self._settings_summary(settings))
        self._thread = threading.Thread(target=self._run, args=(settings,), daemon=True, name="mnist-evidence-optuna")
        self._thread.start()

    def _run(self, settings):
        try:
            self.study, self.results = run_mnist_evidence_optuna_search(
                self.workbench,
                n_trials=settings["n_trials"],
                train_per_digit=settings["train_per_digit"],
                eval_per_digit=settings["eval_per_digit"],
                epochs=settings["epochs"],
                batch_size=settings["batch_size"],
                seed=settings["seed"],
                optimize_reservoir=settings["optimize_reservoir"],
                run_no_recurrence_ablation=settings["run_no_recurrence_ablation"],
                readout_exclude_input=settings["readout_exclude_input"],
                readout_neighborhood_hops=settings["readout_neighborhood_hops"],
                stop_event=self._stop,
                status_callback=lambda text: self._set_status(text + " | active " + self._settings_summary(settings)),
            )
            self.table.value = _mnist_optuna_results_table(self.results)
            best = self.results[0]["score"] if self.results else np.nan
            finished = len([t for t in self.study.trials if t.state.is_finished()]) if self.study is not None else len(self.results)
            complete = len(_complete_mnist_trials(self.study)) if self.study is not None else len(self.results)
            pruned = len([t for t in self.study.trials if t.state == optuna.trial.TrialState.PRUNED]) if self.study is not None else 0
            self._set_status(f"done | finished={finished}/{settings['n_trials']} complete={complete} pruned={pruned} valid={len(self.results)} | best={best:.4f}")
        except Exception as exc:
            self._set_status(f"Optuna search failed: {type(exc).__name__}: {exc}")
        finally:
            self._active_settings = None
            self._set_running(False)

    def stop(self):
        if self._thread is None or not self._thread.is_alive():
            self._set_status("no search running")
            return
        self._stop.set()
        suffix = "" if self._active_settings is None else f" | active {self._settings_summary(self._active_settings)}"
        self._set_status("stop requested" + suffix)

    def apply_best(self):
        install_policy = bool(self.install_policy.value)
        valid_results = [
            row for row in self.results
            if _is_finite_number(row.get("eval_metrics", {}).get("accuracy", np.nan))
            and (not install_policy or row.get("policy_state") is not None)
        ]
        if not valid_results:
            self._set_status("no completed Optuna result with final metrics to apply")
            return
        apply_mnist_evidence_optuna_result(valid_results[0], self.workbench, install_policy=install_policy)
        self._set_status(f"applied trial {valid_results[0]['trial']} | score={valid_results[0]['score']:.4f}")

    def display(self):
        display(widgets.VBox([
            widgets.HBox([self.run_button, self.stop_button, self.apply_button, self.optimize_reservoir, self.no_recurrence_ablation, self.install_policy]),
            widgets.HBox([self.mask_input_readout, self.mask_hops]),
            widgets.HBox([self.trials, self.train_per_digit, self.eval_per_digit, self.epochs, self.batch_size, self.seed]),
            self.status,
            self.table,
        ]))


if widgets is not None and "evidence_workbench" in globals():
    _previous_evidence_optuna_panel = globals().get("evidence_optuna_panel")
    if _previous_evidence_optuna_panel is not None:
        _thread = getattr(_previous_evidence_optuna_panel, "_thread", None)
        if _thread is not None and _thread.is_alive():
            _previous_evidence_optuna_panel.stop()
    evidence_optuna_panel = EvidenceOptunaSearchPanel(evidence_workbench)
    evidence_optuna_panel.display()
elif widgets is not None:
    print("Run the Evidence Accumulator Workbench cell before this Optuna panel.")


In [ ]:
apply_mnist_evidence_optuna_result(
      evidence_optuna_panel.results[0],
      evidence_workbench,
      install_policy=True,
  )


## Notes For Larger Runs

- `record_stride` is not used here; features are sampled directly from engine state.
- To inspect raw dynamics, use `engine.start_static_record(...)` with the memory-safe streaming path.
- Feature sampling uses the CUDA reservoir feature kernel and transfers one averaged feature vector per image.
- The probe feature vector is reservoir-only: spike trace and membrane voltage. Pixels are never appended to the classifier input.
